<a href="https://colab.research.google.com/github/Kuo1204/114-1KUO-REPO-/blob/main/PPT2Course_AI_Final_v15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get -qq update
!apt-get -qq install -y libreoffice poppler-utils ffmpeg \
    fonts-noto-cjk fonts-noto-cjk-extra \
    fonts-crosextra-carlito fonts-crosextra-caladea fonts-liberation
print('✅ 系統軟體與字型安裝完成')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.12_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.12_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unselected package libuno-sal3.
Preparing to unpack .../002-libuno-sal3_1%3a7.3.7-0ubuntu0.22.04.12_amd64.deb ...
Unpacking libuno-sal3 (1:7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unsele

In [2]:
!pip -q install gradio python-pptx pdf2image python-docx reportlab edge-tts google-genai python-dotenv pypdf
print('✅ Python 套件安裝完成')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.1 MB/s eta 0:00:00
✅ Python 套件安裝完成


In [3]:
%%writefile utils.py
# -*- coding: utf-8 -*-
"""
utils.py
========
共用工具模組。

提供整個 PPT2Course AI 系統會用到的基礎功能：
1. 專案資料夾路徑管理 (PPT / Script / Audio / Images / Output / Assets)
2. 統一的 Logger（同時輸出到終端機 與 提供給 Gradio 即時 Log 使用的字串佇列）
3. 環境檢查（LibreOffice / FFmpeg 是否存在）
4. 常用小工具（清空資料夾、取得安全檔名等）

之後所有模組 (ppt_reader.py / script_generator.py / tts.py / subtitle.py / video.py / app.py)
都會 import 這個檔案。
"""

import os
import re
import shutil
import logging
import subprocess
from datetime import datetime

# ------------------------------------------------------------
# 1. 專案根目錄與各功能資料夾路徑
# ------------------------------------------------------------
# BASE_DIR：本檔案所在目錄，作為整個專案的根目錄。
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# 定義六大工作資料夾。
DIR_PPT = os.path.join(BASE_DIR, "PPT")          # 使用者上傳的原始 PPT
DIR_SCRIPT = os.path.join(BASE_DIR, "Script")    # 產生/上傳的講稿 (docx, txt)
DIR_AUDIO = os.path.join(BASE_DIR, "Audio")      # AI 配音產生的 mp3/wav
DIR_IMAGES = os.path.join(BASE_DIR, "Images")    # PPT 每頁轉出的圖片
DIR_OUTPUT = os.path.join(BASE_DIR, "Output")    # 最終輸出 (mp4, srt, docx, pdf)
DIR_ASSETS = os.path.join(BASE_DIR, "Assets")    # 使用者上傳的 Logo / 背景音樂 / 片頭片尾

ALL_DIRS = [DIR_PPT, DIR_SCRIPT, DIR_AUDIO, DIR_IMAGES, DIR_OUTPUT, DIR_ASSETS]


def ensure_all_dirs():
    """
    確保六大工作資料夾都存在。
    若資料夾不存在則自動建立。
    在 app.py 啟動時應呼叫一次。
    """
    for d in ALL_DIRS:
        os.makedirs(d, exist_ok=True)


def make_session_dir(base_dir: str, session_id: str = None) -> str:
    """
    在指定的 base_dir 底下建立一個「本次工作階段」專屬的子資料夾。
    用途：避免多個使用者/多次上傳的檔案互相覆蓋。

    參數:
        base_dir: 例如 DIR_IMAGES 或 DIR_AUDIO
        session_id: 若未提供，會用目前時間戳記自動產生 (YYYYMMDD_HHMMSS)

    回傳:
        建立好的資料夾完整路徑
    """
    if session_id is None:
        session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_path = os.path.join(base_dir, session_id)
    os.makedirs(session_path, exist_ok=True)
    return session_path


def clear_dir(dir_path: str):
    """
    清空指定資料夾內的所有檔案與子資料夾（但保留該資料夾本身）。
    用於每次重新產生課程前，避免舊檔案殘留造成混淆。
    """
    if not os.path.exists(dir_path):
        os.makedirs(dir_path, exist_ok=True)
        return
    for name in os.listdir(dir_path):
        full_path = os.path.join(dir_path, name)
        if os.path.isdir(full_path):
            shutil.rmtree(full_path)
        else:
            os.remove(full_path)


def safe_filename(filename: str) -> str:
    """
    將檔名中可能造成路徑問題的字元移除，回傳安全的檔名。
    例如使用者上傳的檔名包含空白、特殊符號時使用。
    """
    keep_chars = (" ", ".", "_", "-")
    return "".join(c for c in filename if c.isalnum() or c in keep_chars).strip()


def natural_sort_key(path: str):
    """
    「自然排序」用的 key 函式：把檔名中的數字當成整數比較，而不是逐字元比較文字。

    用途：使用者自行匯出投影片圖片時，檔名常見類似 "投影片1.png" "投影片2.png" ...
    "投影片10.png" 這種格式。如果用一般文字排序，會出現
    "投影片1.png" < "投影片10.png" < "投影片2.png" 這種錯誤順序
    （因為文字排序時 "1" < "2"，但 "10" 這個字串本身也是 "1" 開頭，排序會排到 "2" 前面）。
    使用這個 key 排序，才能確保依照「頁碼數字大小」正確排序，而不是依照文字順序。
    """
    basename = os.path.basename(path)
    parts = re.split(r"(\d+)", basename)
    return [int(p) if p.isdigit() else p.lower() for p in parts]


# ------------------------------------------------------------
# 2. Logger 設定
# ------------------------------------------------------------
# 建立一個全域 logger，同時：
#   (a) 印到終端機，方便本機/Colab 開發時除錯
#   (b) 寫入 LOG_MESSAGES 這個 list，讓 Gradio 介面可以輪詢並顯示「即時 Log」
LOG_MESSAGES = []  # Gradio 介面會定期讀取這個 list 來更新畫面上的 Log 區塊

logger = logging.getLogger("ppt2course")
logger.setLevel(logging.INFO)

if not logger.handlers:
    _console_handler = logging.StreamHandler()
    _console_handler.setFormatter(
        logging.Formatter("[%(asctime)s] [%(levelname)s] %(message)s", "%H:%M:%S")
    )
    logger.addHandler(_console_handler)


def log(message: str, level: str = "info"):
    """
    統一的紀錄函式。所有模組都應該呼叫這個函式，而不是直接 print()。

    參數:
        message: 要記錄的訊息文字
        level: "info" / "warning" / "error"

    效果:
        1. 依照 level 呼叫對應的 logging 函式（會顯示在終端機）
        2. 將帶時間戳記的訊息加進 LOG_MESSAGES，供 Gradio 前端顯示
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    line = f"[{timestamp}] {message}"
    LOG_MESSAGES.append(line)

    if level == "warning":
        logger.warning(message)
    elif level == "error":
        logger.error(message)
    else:
        logger.info(message)


def get_log_text() -> str:
    """
    回傳目前累積的所有 Log 訊息，串成單一字串（換行分隔）。
    給 Gradio 的 Textbox / Markdown 元件顯示用。
    """
    return "\n".join(LOG_MESSAGES)


def clear_log():
    """清空目前的 Log 紀錄，通常在每次開始新的一次課程生成時呼叫。"""
    LOG_MESSAGES.clear()


# ------------------------------------------------------------
# 3. 環境檢查（LibreOffice / FFmpeg）
# ------------------------------------------------------------
def check_command_exists(command: str) -> bool:
    """
    檢查系統上是否安裝了某個命令列工具（例如 soffice、ffmpeg）。
    使用 shutil.which，跨平台 (Windows / Linux / Colab) 皆可運作。
    """
    return shutil.which(command) is not None


def check_environment() -> dict:
    """
    檢查系統執行 PPT2Course AI 所需的外部工具是否齊全。

    回傳一個 dict，例如：
    {
        "libreoffice": True,
        "ffmpeg": True,
        "all_ok": True
    }

    使用時機：
        app.py 啟動時呼叫一次，若缺少工具則在畫面上提示使用者，
        避免執行到一半才失敗，讓使用者不知道原因。
    """
    # Windows 上 LibreOffice 命令通常是 soffice.exe，但 shutil.which 對兩者都適用。
    has_libreoffice = check_command_exists("soffice") or check_command_exists("libreoffice")
    has_ffmpeg = check_command_exists("ffmpeg")

    result = {
        "libreoffice": has_libreoffice,
        "ffmpeg": has_ffmpeg,
        "all_ok": has_libreoffice and has_ffmpeg,
    }

    if not has_libreoffice:
        log("⚠️ 找不到 LibreOffice (soffice)，PPT 轉圖片功能將無法使用。", "warning")
    if not has_ffmpeg:
        log("⚠️ 找不到 FFmpeg，影片合成功能將無法使用。", "warning")

    return result


def run_subprocess(cmd_list: list, description: str = "") -> subprocess.CompletedProcess:
    """
    統一的外部命令執行函式（例如呼叫 soffice / ffmpeg）。
    會自動記錄 log，並在失敗時記錄錯誤內容，方便除錯。

    參數:
        cmd_list: 命令列參數列表，例如 ["soffice", "--headless", "--convert-to", "pdf", "a.pptx"]
        description: 給 log 用的說明文字，例如 "轉換 PPT 為 PDF"

    回傳:
        subprocess.CompletedProcess 物件
    """
    if description:
        log(f"執行中：{description}")

    result = subprocess.run(
        cmd_list,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        log(f"❌ 命令執行失敗：{' '.join(cmd_list)}\n{result.stderr}", "error")
    else:
        log(f"✅ 完成：{description or ' '.join(cmd_list)}")

    return result


if __name__ == "__main__":
    # 簡單自我測試：確保資料夾建立、Log、環境檢查都正常運作。
    ensure_all_dirs()
    log("utils.py 自我測試開始")
    env = check_environment()
    print("環境檢查結果：", env)
    print("目前 Log：")
    print(get_log_text())


Writing utils.py


In [4]:
%%writefile ppt_reader.py
# -*- coding: utf-8 -*-
"""
ppt_reader.py
=============
負責所有「讀取 PowerPoint 檔案」相關的工作：

1. 使用 python-pptx 讀取每一頁投影片的：
   - 文字內容 (標題、內文)
   - 講者備忘稿 (Speaker Notes)
2. 使用 LibreOffice 將 PPT 轉成 PDF，
   再用 pdf2image 把 PDF 每一頁轉成 PNG 圖片
   （這些圖片會被後續 video.py 用來合成影片畫面）

之後 script_generator.py 會使用這裡回傳的文字資料來生成講稿，
video.py 會使用這裡輸出的圖片來合成影片。
"""

import os
import shutil
import tempfile
from pptx import Presentation
from pdf2image import convert_from_path

from utils import log, run_subprocess, check_command_exists


# ------------------------------------------------------------
# 1. 讀取投影片文字內容與備忘稿
# ------------------------------------------------------------
def read_ppt_slides(ppt_path: str) -> list:
    """
    讀取 PPT 檔案，回傳每一頁的文字資訊。

    參數:
        ppt_path: PPT 檔案路徑 (.pptx)

    回傳:
        一個 list，每個元素是一個 dict，代表一頁投影片，例如：
        [
            {
                "slide_index": 1,           # 第幾頁 (從1開始)
                "title": "課程介紹",         # 該頁的標題文字（若有偵測到）
                "body_text": "本課程將...",  # 該頁內文字（不含標題），用換行合併
                "notes": "各位同仁大家好..."  # 該頁的 Speaker Notes（若無則為空字串）
            },
            ...
        ]

    設計理由:
        將標題與內文分開，是因為 AI 生成講稿時（模式二）
        通常需要知道「這頁的主題是什麼」，標題是很好的線索。
    """
    if not os.path.exists(ppt_path):
        raise FileNotFoundError(f"找不到 PPT 檔案: {ppt_path}")

    log(f"開始讀取 PPT 內容：{os.path.basename(ppt_path)}")
    prs = Presentation(ppt_path)

    slides_data = []

    for idx, slide in enumerate(prs.slides, start=1):
        title_text = ""
        body_lines = []
        # 收集「沒被判定為標準標題」的文字框，連同其垂直座標(top)，
        # 供找不到標準標題時，用來猜測最可能是標題的那一個。
        untitled_shapes = []  # list of (top_position, text)

        for shape in slide.shapes:
            if not shape.has_text_frame:
                continue

            text = shape.text_frame.text.strip()
            if not text:
                continue

            # 判斷是否為標題：
            # python-pptx 的 placeholder type 10 (TITLE) 或 13 (CENTER_TITLE) 代表標題文字框。
            is_title_placeholder = False
            if shape.is_placeholder:
                try:
                    ph_type = shape.placeholder_format.type
                    # 用 str 判斷避免不同 python-pptx 版本 enum 對應不到
                    if ph_type is not None and "TITLE" in str(ph_type):
                        is_title_placeholder = True
                except Exception:
                    pass

            if is_title_placeholder and not title_text:
                title_text = text
            else:
                body_lines.append(text)
                # shape.top 是文字框左上角的垂直座標 (EMU 單位)，數字越小代表越靠近頁面上方。
                # 若整頁都沒有標準標題版面，用「最靠上方的文字框」當作標題的猜測依據，
                # 這對很多企業/政府常用的自訂範本（標題其實是一般文字方塊）特別有幫助。
                top_position = shape.top if shape.top is not None else 0
                untitled_shapes.append((top_position, text))

        # 找不到標準標題版面時，改用「最靠頁面上方」的文字框內容當作標題，
        # 並把該行從內文中移除，避免同一段文字在標題和內文重複出現。
        if not title_text and untitled_shapes:
            untitled_shapes.sort(key=lambda item: item[0])
            guessed_title = untitled_shapes[0][1]
            # 標題通常較短（一行左右），避免誤把一大段內文當標題；
            # 若最上方文字框內容過長，就不當作標題使用，維持「(無標題)」，避免誤判。
            if len(guessed_title) <= 40:
                title_text = guessed_title
                body_lines.remove(guessed_title)

        # 讀取 Speaker Notes（講者備忘稿）
        notes_text = ""
        if slide.has_notes_slide:
            notes_slide = slide.notes_slide
            if notes_slide.notes_text_frame is not None:
                notes_text = notes_slide.notes_text_frame.text.strip()

        slide_info = {
            "slide_index": idx,
            "title": title_text,
            "body_text": "\n".join(body_lines),
            "notes": notes_text,
        }
        slides_data.append(slide_info)

    log(f"✅ 共讀取到 {len(slides_data)} 頁投影片內容")
    return slides_data


# ------------------------------------------------------------
# 2. 將 PPT 轉換成每頁圖片 (PPT -> PDF -> PNG)
# ------------------------------------------------------------
def convert_ppt_to_images(ppt_path: str, output_dir: str, dpi: int = 150) -> list:
    """
    將 PPT 檔案的每一頁轉換成 PNG 圖片，供影片合成使用。

    流程:
        1. 呼叫 LibreOffice (soffice --headless) 把 .pptx 轉成 .pdf
        2. 用 pdf2image 把 PDF 逐頁轉成 PNG 圖片，存到 output_dir

    參數:
        ppt_path: PPT 檔案路徑
        output_dir: 圖片輸出資料夾
        dpi: 圖片解析度，數字越大圖片越清晰但檔案越大（預設150已足夠1080p影片使用）

    回傳:
        圖片路徑的 list，依照頁碼順序排列，例如：
        ["Images/xxx/slide_01.png", "Images/xxx/slide_02.png", ...]

    關於「PPT 轉換後跑版」的說明與處理：
        跑版最常見的兩個原因，這裡都有處理：
        1. 【字型缺失】PPT 使用的字型（例如 Calibri、Cambria 等 Office 預設字型）
           如果系統沒有安裝，LibreOffice 會自動換成其他字型，寬度不同就會造成
           文字換行位置跑掉、超出文字框。解法是安裝「metric-compatible」的替代字型
           （見 requirements/Colab 安裝指令中的 fonts-crosextra-carlito 等），
           讓替代字型的字寬跟原本的 Office 字型幾乎一致。
        2. 【LibreOffice 設定檔衝突】同一個系統如果連續、重複呼叫 soffice 轉換多份
           PPT，共用同一份使用者設定檔（profile）有時會互相干擾，導致某次轉換
           不完整或版面跑掉。這裡改成「每次轉換都使用獨立的暫存設定檔」
           (-env:UserInstallation)，並在轉換失敗時自動重試一次，大幅降低這類問題。
    """
    if not check_command_exists("soffice") and not check_command_exists("libreoffice"):
        raise EnvironmentError(
            "找不到 LibreOffice，無法將 PPT 轉換成圖片。"
            "請先安裝 LibreOffice（Windows/Colab/HF Spaces 安裝方式請參考 README）。"
        )

    os.makedirs(output_dir, exist_ok=True)

    # --- Step 1: PPT -> PDF ---
    soffice_cmd = "soffice" if check_command_exists("soffice") else "libreoffice"
    log(f"轉換 PPT 為 PDF 中（使用 {soffice_cmd}）...")

    ppt_basename = os.path.splitext(os.path.basename(ppt_path))[0]
    pdf_path = os.path.join(output_dir, f"{ppt_basename}.pdf")

    max_attempts = 2
    for attempt in range(1, max_attempts + 1):
        # 每次轉換都用「獨立、全新」的 LibreOffice 使用者設定檔，
        # 避免多次轉換共用同一份 profile 造成互相干擾、版面跑掉的問題。
        profile_dir = tempfile.mkdtemp(prefix="lo_profile_")
        try:
            run_subprocess(
                [
                    soffice_cmd,
                    "--headless",
                    "--norestore",
                    f"-env:UserInstallation=file://{profile_dir}",
                    "--convert-to",
                    "pdf:impress_pdf_Export",
                    "--outdir",
                    output_dir,
                    ppt_path,
                ],
                description=f"PPT 轉換為 PDF（第 {attempt}/{max_attempts} 次嘗試）",
            )
        finally:
            shutil.rmtree(profile_dir, ignore_errors=True)

        if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 0:
            break

        if attempt < max_attempts:
            log("⚠️ PPT 轉換 PDF 結果異常，重新嘗試一次...", "warning")
        else:
            raise RuntimeError(f"PPT 轉 PDF 失敗，找不到輸出檔案：{pdf_path}")

    # --- Step 2: PDF -> PNG (逐頁) ---
    log("將 PDF 逐頁轉換為圖片中...")
    pages = convert_from_path(pdf_path, dpi=dpi)

    image_paths = []
    for i, page in enumerate(pages, start=1):
        image_path = os.path.join(output_dir, f"slide_{i:03d}.png")
        page.save(image_path, "PNG")
        image_paths.append(image_path)

    log(f"✅ 共產生 {len(image_paths)} 張投影片圖片，存放於 {output_dir}")
    return image_paths


# ------------------------------------------------------------
# 3. 整合函式：一次讀取文字 + 產生圖片
# ------------------------------------------------------------
def load_presentation(ppt_path: str, image_output_dir: str, dpi: int = 150) -> dict:
    """
    整合函式：同時讀取 PPT 文字內容，並轉換出每頁圖片。
    這是提供給 app.py 呼叫的主要入口函式。

    回傳:
        {
            "slides_text": [...],     # read_ppt_slides() 的結果
            "slides_images": [...],   # convert_ppt_to_images() 的結果
            "slide_count": 10
        }
    """
    slides_text = read_ppt_slides(ppt_path)
    slides_images = convert_ppt_to_images(ppt_path, image_output_dir, dpi=dpi)

    if len(slides_text) != len(slides_images):
        log(
            f"⚠️ 注意：文字頁數({len(slides_text)}) 與 圖片頁數({len(slides_images)}) 不一致，"
            "請確認 PPT 檔案是否正常。",
            "warning",
        )

    return {
        "slides_text": slides_text,
        "slides_images": slides_images,
        "slide_count": len(slides_text),
    }


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    直接執行本檔案時，會：
    1. 建立一份範例 PPT（3頁，含標題/內文/備忘稿）
    2. 測試文字讀取功能
    3. 測試轉圖片功能
    用來驗證整個模組在目前環境下可以正常運作。
    """
    from pptx.util import Inches

    test_ppt_path = "/home/claude/PPT2Course_AI/PPT/_test_sample.pptx"
    test_image_dir = "/home/claude/PPT2Course_AI/Images/_test_sample"

    # 建立測試用 PPT
    prs = Presentation()
    layout = prs.slide_layouts[1]  # 標題 + 內文版面

    sample_slides = [
        ("課程介紹", "本課程將說明公司資訊安全政策的重要性。", "各位同仁大家好，歡迎參加本次教育訓練課程。"),
        ("資安法規概述", "個人資料保護法\n資通安全管理法", ""),
        ("結語與Q&A", "感謝大家的參與，如有問題歡迎提出。", "課程到此結束，謝謝大家。"),
    ]

    for title, body, notes in sample_slides:
        slide = prs.slides.add_slide(layout)
        slide.shapes.title.text = title
        slide.placeholders[1].text = body
        if notes:
            slide.notes_slide.notes_text_frame.text = notes

    prs.save(test_ppt_path)
    print(f"已建立測試 PPT：{test_ppt_path}")

    # 測試整合函式
    result = load_presentation(test_ppt_path, test_image_dir)

    print("\n===== 文字讀取結果 =====")
    for s in result["slides_text"]:
        print(s)

    print("\n===== 圖片產生結果 =====")
    for p in result["slides_images"]:
        print(p, "存在:", os.path.exists(p))


Writing ppt_reader.py


In [5]:
%%writefile script_cleaner.py
# -*- coding: utf-8 -*-
"""
script_cleaner.py
==================
負責「送進 TTS 之前」的文字清理工作。

問題背景：
    使用者的講稿（不論是自己寫的 Word/txt、或 AI 生成的講稿）常常會包含排版用的
    分隔線或空白，例如：
        __________
        ------
        ......
        ***
        ===
    這些符號本身不是要「唸出來」的內容，只是視覺上的分隔線；但部分 TTS 引擎會把
    它們當成文字逐字朗讀（例如把一整行底線唸成「底線、底線、底線」），聽起來非常怪。

解法：
    提供統一的 clean_script(text) 函式，在文字送進 tts.py 的 edge-tts 合成之前，
    先過濾掉這些「非朗讀用」的雜訊行與多餘空白。所有 TTS 相關流程都呼叫這一個函式，
    不要各自實作各自的清理邏輯（避免規則分散、之後改一個地方漏掉其他地方）。

清理規則（對應需求文件）：
    1. 移除完全空白的行
    2. 移除只有空白字元(含全形空白)的行
    3. Tab 轉成空白
    4. 連續空白收斂成一個空白（例如 "Hello      World" -> "Hello World"）
    5. 只有底線的行（一個以上連續 "_"）整行刪除
    6. 只有減號的行（一個以上連續 "-"）整行刪除
    7. 只有句點/省略號的行（"...", "……" 等）整行刪除
    8. Markdown 分隔線（***、---、___）整行刪除（規則5/6已涵蓋，*** 另外處理）
    9. 整行只由特殊符號組成（###、///、===、~~~ 等）整行刪除
    10. 清理後不會出現連續多個空白行（段落之間最多保留一行）
"""

import re


# ------------------------------------------------------------
# 判斷「整行都是非朗讀用符號」的規則
# ------------------------------------------------------------
# 只由底線 / 減號 / 句點(含全形省略號…) 其中一種符號重複組成的行，整行視為分隔線。
_PURE_SEPARATOR_PATTERN = re.compile(r"^(_+|-+|\.+|…+)$")

# 整行只由「非文字符號」組成（不含任何英數字、中文、日文、韓文），
# 例如 "###"、"///"、"==="、"~~~"、"***" 這類分隔/裝飾符號。
_SYMBOL_ONLY_PATTERN = re.compile(
    r"^[^\w\u4e00-\u9fff\u3040-\u30ff\u31f0-\u31ff\uac00-\ud7a3]+$"
)

# 連續空白字元（含全形空白 \u3000、Tab 已提前轉換），收斂成單一半形空白。
_MULTI_SPACE_PATTERN = re.compile(r"[ \u3000]+")

# 清理完成後，避免殘留 2 行以上的連續空白行。
_MULTI_BLANK_LINE_PATTERN = re.compile(r"\n{3,}")


def _is_junk_line(stripped_line: str) -> bool:
    """
    判斷「已去除頭尾空白」的一行文字，是不是應該被整行刪除的雜訊行。
    """
    if not stripped_line:
        return True  # 空白行 / 只有空格的行

    if _PURE_SEPARATOR_PATTERN.match(stripped_line):
        return True  # 只有底線 / 減號 / 句點的行

    if _SYMBOL_ONLY_PATTERN.match(stripped_line):
        return True  # 整行只有特殊符號（###、///、***、=== 等）

    return False


def clean_script(text: str) -> str:
    """
    清理講稿文字，移除不該被 TTS 朗讀出來的排版分隔符號與多餘空白。

    參數:
        text: 原始講稿文字（可能包含分隔線、多餘空白、Tab 等）

    回傳:
        清理後的文字，可以直接送進 TTS 合成語音。

    範例:
        >>> clean_script("今天介紹 AI\\n_________\\n下一章開始")
        '今天介紹 AI\\n下一章開始'
    """
    if not text:
        return ""

    # Tab 一律轉成半形空白，方便後續統一收斂處理
    text = text.replace("\t", " ")

    cleaned_lines = []
    for raw_line in text.split("\n"):
        stripped = raw_line.strip()

        if _is_junk_line(stripped):
            continue

        # 連續空白（含全形空白）收斂成一個半形空白
        normalized = _MULTI_SPACE_PATTERN.sub(" ", stripped)
        cleaned_lines.append(normalized)

    result = "\n".join(cleaned_lines)

    # 保險：就算前面的邏輯有殘留，這裡再收斂一次，避免出現大量連續空白行
    result = _MULTI_BLANK_LINE_PATTERN.sub("\n\n", result)

    return result.strip()


def clean_scripts(texts: list) -> list:
    """
    批次版本：對「每頁講稿」清單逐一清理，回傳長度相同的清理後清單。
    方便 tts.py 在批次產生配音前，一次呼叫清理整份講稿。
    """
    return [clean_script(t) for t in texts]


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    test_cases = [
        # (輸入, 預期輸出)
        ("今天介紹 AI\n_________\n下一章開始", "今天介紹 AI\n下一章開始"),
        ("Hello      World", "Hello World"),
        ("第一段\n\n\n\n第二段", "第一段\n第二段"),
        ("重點一\n------\n重點二\n......\n重點三", "重點一\n重點二\n重點三"),
        ("###\n主題\n///\n內容\n===", "主題\n內容"),
        ("　　全形空白測試　　結果", "全形空白測試 結果"),
        ("***\n---\n___\n", ""),
        ("", ""),
        ("單純一行文字，沒有任何符號問題。", "單純一行文字，沒有任何符號問題。"),
        ("重點：\n\t這裡有Tab\t結尾", "重點：\n這裡有Tab 結尾"),
    ]

    all_passed = True
    for i, (input_text, expected) in enumerate(test_cases, start=1):
        result = clean_script(input_text)
        status = "✅" if result == expected else "❌"
        if result != expected:
            all_passed = False
        print(f"{status} 測試{i}: 輸入={input_text!r}")
        print(f"       輸出={result!r}")
        print(f"       預期={expected!r}")

    print()
    if all_passed:
        print("✅ 全部測試通過")
    else:
        print("❌ 有測試失敗，請檢查上方結果")
        raise SystemExit(1)

    # 批次版本測試
    batch_result = clean_scripts(["today\n___\nline2", "clean text"])
    print("\n批次清理測試:", batch_result)
    assert batch_result == ["today\nline2", "clean text"]
    print("✅ 批次清理測試通過")


Writing script_cleaner.py


In [6]:
%%writefile script_generator.py
# -*- coding: utf-8 -*-
"""
script_generator.py
====================
負責「講稿生成」的四種模式：

模式一 (own)      ：使用者提供 docx/txt 講稿，依頁碼標記自動對應每一頁投影片
模式二 (ai_auto)  ：AI 根據投影片文字，自動生成講稿 (使用 Gemini API)
模式三 (notes)    ：直接使用 PPT 內建的 Speaker Notes 當作講稿
模式四 (ai_polish)：AI 修飾使用者提供的講稿，讓語氣更口語自然

所有模式最終都會回傳「長度 = 投影片頁數」的講稿清單 (list[str])，
方便後續 tts.py 依序拿去產生配音。
"""

import os
import re
import time
from docx import Document

from utils import log


# ------------------------------------------------------------
# 參考資料讀取 (讓 AI 依據真實資料生成/修飾講稿，避免內容失真)
# ------------------------------------------------------------
def extract_text_from_pdf(pdf_path: str) -> str:
    """讀取 PDF 檔案的純文字內容（逐頁擷取後合併）。"""
    from pypdf import PdfReader

    reader = PdfReader(pdf_path)
    texts = []
    for page in reader.pages:
        texts.append(page.extract_text() or "")
    return "\n".join(texts)


def extract_text_from_reference_docx(docx_path: str) -> str:
    """讀取 docx 參考資料的純文字內容。"""
    doc = Document(docx_path)
    return "\n".join(p.text for p in doc.paragraphs)


def extract_text_from_reference_txt(txt_path: str) -> str:
    """讀取 txt 參考資料的純文字內容。"""
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def extract_reference_text(file_paths: list, max_chars_per_file: int = 20000) -> str:
    """
    讀取一份或多份「參考資料」檔案 (PDF / docx / txt)，合併成一段文字，
    供 AI 生成或修飾講稿時，當作「真實資料依據」放進 prompt 一起參考。

    參數:
        file_paths: 參考資料檔案路徑清單
        max_chars_per_file: 每份檔案最多擷取的字數，避免單一檔案過長
                            導致整段 prompt 太大（Gemini 雖然容許很長的內容，
                            但過長仍會拖慢速度、增加費用），超過會自動截斷。

    回傳:
        合併後的參考資料文字（每份資料前會標註來源檔名），
        若沒有提供任何檔案，回傳空字串。
    """
    if not file_paths:
        return ""

    blocks = []
    for path in file_paths:
        if not path:
            continue
        ext = os.path.splitext(path)[1].lower()
        name = os.path.basename(path)

        try:
            if ext == ".pdf":
                text = extract_text_from_pdf(path)
            elif ext == ".docx":
                text = extract_text_from_reference_docx(path)
            elif ext == ".txt":
                text = extract_text_from_reference_txt(path)
            else:
                log(f"⚠️ 不支援的參考資料格式，已略過：{name}", "warning")
                continue
        except Exception as e:
            log(f"⚠️ 讀取參考資料失敗，已略過：{name}（原因：{e}）", "warning")
            continue

        text = text.strip()
        if not text:
            log(f"⚠️ 參考資料內容是空的，已略過：{name}", "warning")
            continue

        if len(text) > max_chars_per_file:
            text = text[:max_chars_per_file] + "\n...(內容過長，已自動截斷)"

        blocks.append(f"【參考資料來源：{name}】\n{text}")
        log(f"✅ 已讀取參考資料：{name}（擷取 {len(text)} 字）")

    return "\n\n".join(blocks)


# ------------------------------------------------------------
# 共用：頁碼標記格式
# ------------------------------------------------------------
# 支援的頁碼標記寫法，例如：
#   第1頁 / 第 1 頁 / 第1頁：
#   Slide1 / Slide 1
#   P1
#   投影片1 / 投影片 1
PAGE_MARKER_PATTERN = re.compile(
    r"^(?:第\s*(\d+)\s*頁|[Ss]lide\s*(\d+)|P(\d+)|投影片\s*(\d+))\s*[:：]?\s*$"
)


def format_scripts_for_display(slides_text: list, scripts: list) -> str:
    """
    將「每頁講稿清單」轉換成一份好閱讀、也方便使用者直接編輯的文字內容。
    格式範例：
        ===== 第1頁：課程介紹 =====
        各位同仁大家好...

        ===== 第2頁：資安法規概述 =====
        (講稿內容)

    這段文字會顯示在 Gradio 的編輯框中，使用者可以直接修改文字內容，
    改完後系統會用 parse_scripts_from_display() 重新解析回清單。
    """
    blocks = []
    for s, script in zip(slides_text, scripts):
        header = f"===== 第{s['slide_index']}頁：{s['title'] or '(無標題)'} ====="
        content = script.strip() if script else "(尚無內容，請手動輸入或重新產生)"
        blocks.append(f"{header}\n{content}")
    return "\n\n".join(blocks)


def parse_scripts_from_display(display_text: str, slide_count: int) -> list:
    """
    將使用者在編輯框中修改過的文字，重新解析回「每頁一則」的講稿清單。
    解析規則：以 "===== 第X頁" 開頭的那一行作為分頁標記。
    """
    lines = display_text.split("\n")
    scripts = ["" for _ in range(slide_count)]

    header_pattern = re.compile(r"^=+\s*第(\d+)頁")
    marker_indices = []
    for i, line in enumerate(lines):
        m = header_pattern.match(line.strip())
        if m:
            marker_indices.append((int(m.group(1)), i))

    for idx, (page_num, line_idx) in enumerate(marker_indices):
        start = line_idx + 1
        end = marker_indices[idx + 1][1] if idx + 1 < len(marker_indices) else len(lines)
        content = "\n".join(lines[start:end]).strip()
        if content == "(尚無內容，請手動輸入或重新產生)":
            content = ""
        if 1 <= page_num <= slide_count:
            scripts[page_num - 1] = content

    return scripts


# ------------------------------------------------------------
# 模式一：讀取使用者提供的 docx / txt 講稿
# ------------------------------------------------------------
def _split_lines_by_slide(lines: list, slide_count: int) -> list:
    """
    共用邏輯：把一堆文字行，依照頁碼分割成對應投影片頁數的清單。

    優先順序：
        1. 若偵測到「第X頁」等頁碼標記 -> 依標記精準分割
        2. 若沒有標記 -> 改用「空白行分段」，依序對應第1、2、3...頁
           (適合使用者只是單純每頁空一行寫講稿的情況)
    """
    marker_indices = []
    for i, line in enumerate(lines):
        m = PAGE_MARKER_PATTERN.match(line.strip())
        if m:
            num = next(g for g in m.groups() if g is not None)
            marker_indices.append((int(num), i))

    scripts = ["" for _ in range(slide_count)]

    if marker_indices:
        log(f"偵測到 {len(marker_indices)} 個頁碼標記，依標記分割講稿")
        for idx, (page_num, line_idx) in enumerate(marker_indices):
            start = line_idx + 1
            end = marker_indices[idx + 1][1] if idx + 1 < len(marker_indices) else len(lines)
            content = "\n".join(l for l in lines[start:end] if l.strip()).strip()
            if 1 <= page_num <= slide_count:
                scripts[page_num - 1] = content
            else:
                log(f"⚠️ 講稿中標記的頁碼 {page_num} 超出投影片範圍(共{slide_count}頁)，已略過", "warning")
        return scripts

    log("未偵測到頁碼標記，改用「空白行分段」方式依序對應投影片頁數", "warning")
    blocks = []
    current = []
    for l in lines:
        if l.strip() == "":
            if current:
                blocks.append("\n".join(current).strip())
                current = []
        else:
            current.append(l)
    if current:
        blocks.append("\n".join(current).strip())

    for i in range(slide_count):
        if i < len(blocks):
            scripts[i] = blocks[i]

    if len(blocks) != slide_count:
        log(
            f"⚠️ 講稿段落數({len(blocks)}) 與投影片頁數({slide_count}) 不一致，"
            "建議在講稿中加入「第1頁」「第2頁」等標記，確保對應正確。",
            "warning",
        )

    return scripts


def read_script_from_docx(docx_path: str, slide_count: int) -> list:
    """讀取 .docx 講稿檔案，回傳對應每頁投影片的講稿清單。"""
    log(f"讀取使用者講稿 (Word)：{os.path.basename(docx_path)}")
    doc = Document(docx_path)
    lines = [p.text for p in doc.paragraphs]
    return _split_lines_by_slide(lines, slide_count)


def read_script_from_txt(txt_path: str, slide_count: int) -> list:
    """讀取 .txt 講稿檔案，回傳對應每頁投影片的講稿清單。"""
    log(f"讀取使用者講稿 (txt)：{os.path.basename(txt_path)}")
    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.read().split("\n")
    return _split_lines_by_slide(lines, slide_count)


# ------------------------------------------------------------
# 模式三：直接使用 PPT Speaker Notes
# ------------------------------------------------------------
def get_notes_scripts(slides_text: list) -> list:
    """
    模式三：直接把每頁的 Speaker Notes 當作講稿使用。
    slides_text 來自 ppt_reader.read_ppt_slides() 的回傳結果。
    """
    scripts = []
    empty_count = 0
    for s in slides_text:
        if not s["notes"]:
            empty_count += 1
        scripts.append(s["notes"])

    if empty_count > 0:
        log(f"⚠️ 有 {empty_count} 頁投影片沒有備忘稿內容，該頁講稿會是空白", "warning")
    log("✅ 已讀取所有投影片的 Speaker Notes 作為講稿")
    return scripts


# ------------------------------------------------------------
# 模式二 & 模式四：呼叫 Gemini API
# ------------------------------------------------------------
def _extract_retry_delay(error_str: str, default_seconds: int = 20) -> int:
    """
    嘗試從 Gemini API 回傳的錯誤訊息中，解析出建議的重試等待秒數 (retryDelay)。
    Google 免費方案常見的 429 額度超過錯誤，會在訊息中附上建議等待秒數，
    例如 "retryDelay": "36s"。解析失敗則使用預設值，並多留 2 秒緩衝。
    """
    m = re.search(r"retryDelay['\"]?\s*[:=]\s*['\"]?(\d+)", error_str)
    if m:
        return int(m.group(1)) + 2
    return default_seconds


def _call_gemini(
    prompt: str,
    api_key: str,
    model_name: str = "gemini-flash-latest",
    max_retries: int = 5,
) -> str:
    """
    共用函式：呼叫 Gemini API 並回傳純文字結果。

    使用 Google 官方新版統一 SDK「google-genai」（套件名稱 google-genai，
    import 時寫作 `from google import genai`）。
    舊版 google-generativeai 套件已於 2025/8/31 由 Google 正式棄用。

    模型名稱使用 "gemini-flash-latest"：這是 Google 提供的「別名」模型，
    會自動指向 Google 目前最新、最推薦的 Flash 模型（例如目前指向 gemini-3.5-flash）。
    Google 這幾個月頻繁淘汰舊模型 (gemini-1.5-flash / gemini-2.5-flash 陸續下架)，
    使用別名可以避免之後模型改版又要重新修改程式碼。

    免費方案的 API Key 有嚴格的「每分鐘請求數」額度限制，投影片頁數較多時
    很容易在中段觸發 429 (RESOURCE_EXHAUSTED) 錯誤。這裡加入自動重試機制：
    遇到額度限制時，會依照 Google 建議的等待秒數自動暫停後重試，最多重試
    max_retries 次，讓整批講稿仍然可以順利跑完，不需要使用者手動介入。

    之後若要切換其他 LLM (例如 OpenAI / Claude)，只需要修改這個函式即可，
    不影響上層的四種模式邏輯。
    """
    from google import genai

    client = genai.Client(api_key=api_key)

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(model=model_name, contents=prompt)
            return (response.text or "").strip()
        except Exception as e:
            last_error = e
            error_str = str(e)
            is_rate_limit = "RESOURCE_EXHAUSTED" in error_str or "429" in error_str
            if is_rate_limit and attempt < max_retries:
                wait_seconds = _extract_retry_delay(error_str)
                log(
                    f"⏳ 觸發免費方案額度限制，等待 {wait_seconds} 秒後自動重試"
                    f"（第 {attempt}/{max_retries} 次）...",
                    "warning",
                )
                time.sleep(wait_seconds)
                continue
            raise last_error

    raise last_error


def generate_script_with_ai(
    slide_info: dict,
    api_key: str,
    reference_text: str = "",
    model_name: str = "gemini-flash-latest",
) -> str:
    """
    模式二：根據單一頁投影片的文字內容，請 AI 生成一段自然口語的旁白講稿。

    參數:
        reference_text: 選填。使用者上傳的參考資料合併文字
                        (由 extract_reference_text() 產生)。
                        若有提供，AI 會被要求「依據這份真實資料」補充細節，
                        並被明確禁止編造資料中沒有的內容，藉此提升講稿的真實性。
    """
    slide_content = slide_info["title"]
    if slide_info["body_text"]:
        slide_content += "\n" + slide_info["body_text"]

    reference_block = ""
    if reference_text:
        reference_block = f"""
以下是與本課程主題相關的參考資料，請優先根據這些真實資料來補充正確的細節、數字、
法規條文、案例或專有名詞。絕對不可以編造參考資料中沒有出現的內容；
如果投影片內容與參考資料都沒有提到的細節，就不要憑空補充。

--- 參考資料開始 ---
{reference_text}
--- 參考資料結束 ---
"""

    prompt = f"""你是一位企業教育訓練講師。
請依照以下投影片內容，撰寫約30~60秒的口語旁白講稿。
{reference_block}
要求：
- 不要逐字朗讀投影片文字，要用自然口語表達，像是真人講師在課堂上說話
- 保留法規名稱、專有名詞的正確用字，不可竄改
- 使用繁體中文
- 只輸出講稿本身文字，不要加任何標題、說明或引號

投影片內容：
{slide_content}
"""
    try:
        return _call_gemini(prompt, api_key, model_name)
    except Exception as e:
        log(f"❌ 第 {slide_info['slide_index']} 頁 AI 講稿生成失敗：{e}", "error")
        return ""


def polish_script_with_ai(
    original_text: str,
    api_key: str,
    reference_text: str = "",
    model_name: str = "gemini-flash-latest",
) -> str:
    """
    模式四：請 AI 將使用者提供的講稿修飾得更口語自然，但不改變原意。

    參數:
        reference_text: 選填。若有提供參考資料，AI 在修飾語氣的同時，
                        也可以核對/補充講稿中提到的細節是否與參考資料一致，
                        但同樣不可編造參考資料中沒有的內容。
    """
    reference_block = ""
    if reference_text:
        reference_block = f"""
以下是與本課程主題相關的參考資料，修飾講稿時可以參考這些資料確認用詞正確性，
若原始講稿有可以用參考資料補充得更精確的地方（例如法規全名、數字），可以順手修正，
但絕對不可以編造參考資料中沒有出現的內容：

--- 參考資料開始 ---
{reference_text}
--- 參考資料結束 ---
"""

    prompt = f"""你是一位教育訓練老師的講稿修飾助手。
請將以下講稿修飾成自然口語，適合教育訓練老師實際講課使用。
{reference_block}
規則：
- 絕對不要改變原意，不要增加或刪減重要資訊（例如法規名稱、數字、條文）
- 只是讓語氣更自然、更口語化，去除生硬的書面用語
- 使用繁體中文
- 只輸出修飾後的講稿文字，不要加任何說明或標題

原始講稿：
{original_text}
"""
    try:
        return _call_gemini(prompt, api_key, model_name)
    except Exception as e:
        log(f"❌ AI 講稿修飾失敗：{e}", "error")
        return original_text  # 修飾失敗時，保留原始講稿，避免內容遺失


# ------------------------------------------------------------
# 統一入口：依模式產生所有頁面的講稿
# ------------------------------------------------------------
def generate_all_scripts(
    mode: str,
    slides_text: list,
    uploaded_script_path: str = None,
    api_key: str = None,
    reference_paths: list = None,
    progress_callback=None,
) -> list:
    """
    依照選擇的旁白模式，產生「每一頁」的講稿清單。

    參數:
        mode: "own" | "ai_auto" | "notes" | "ai_polish"
        slides_text: ppt_reader.read_ppt_slides() 的回傳結果
        uploaded_script_path: 模式一/模式四需要，使用者上傳的 docx 或 txt 路徑
        api_key: 模式二/模式四需要，Gemini API Key
        reference_paths: 選填，模式二/模式四可用。使用者上傳的「參考資料」檔案路徑清單
                         (PDF/docx/txt)，AI 生成或修飾講稿時會依據這些真實資料補充細節，
                         提升講稿內容的真實性、降低憑空杜撰的風險。
        progress_callback: 選填，function(fraction: float) 用來回報進度給 Gradio 進度條

    回傳:
        list[str]，長度 = 投影片頁數
    """
    slide_count = len(slides_text)

    if mode == "notes":
        return get_notes_scripts(slides_text)

    if mode == "own":
        if not uploaded_script_path:
            raise ValueError("模式一（使用自己的講稿）需要先上傳 docx 或 txt 檔案")
        return _read_uploaded_script(uploaded_script_path, slide_count)

    if mode == "ai_auto":
        if not api_key:
            raise ValueError("模式二（AI 自動生成講稿）需要先輸入 Gemini API Key")

        reference_text = extract_reference_text(reference_paths) if reference_paths else ""
        if reference_text:
            log("📚 已讀入參考資料，AI 生成講稿時會依據這些資料補充真實細節")

        scripts = []
        for i, s in enumerate(slides_text):
            log(f"AI 生成講稿中：第 {s['slide_index']}/{slide_count} 頁")
            scripts.append(generate_script_with_ai(s, api_key, reference_text=reference_text))
            if progress_callback:
                progress_callback((i + 1) / slide_count)
            if i < slide_count - 1:
                time.sleep(2)  # 主動間隔，降低免費方案觸發「每分鐘請求數」限制的機率
        log("✅ AI 講稿自動生成完成")
        return scripts

    if mode == "ai_polish":
        if not uploaded_script_path:
            raise ValueError("模式四（AI 修飾講稿）需要先上傳您自己的 docx 或 txt 講稿")
        if not api_key:
            raise ValueError("模式四（AI 修飾講稿）需要先輸入 Gemini API Key")

        reference_text = extract_reference_text(reference_paths) if reference_paths else ""
        if reference_text:
            log("📚 已讀入參考資料，AI 修飾講稿時會依據這些資料核對真實細節")

        raw_scripts = _read_uploaded_script(uploaded_script_path, slide_count)
        polished = []
        for i, text in enumerate(raw_scripts):
            log(f"AI 修飾講稿中：第 {i + 1}/{slide_count} 頁")
            if text.strip():
                polished.append(polish_script_with_ai(text, api_key, reference_text=reference_text))
            else:
                polished.append("")
            if progress_callback:
                progress_callback((i + 1) / slide_count)
            if i < slide_count - 1:
                time.sleep(2)  # 主動間隔，降低免費方案觸發「每分鐘請求數」限制的機率
        log("✅ AI 講稿修飾完成")
        return polished

    raise ValueError(f"未知的旁白模式：{mode}")


def _read_uploaded_script(uploaded_script_path: str, slide_count: int) -> list:
    """依副檔名判斷要用 docx 或 txt 的方式讀取使用者上傳的講稿。"""
    ext = os.path.splitext(uploaded_script_path)[1].lower()
    if ext == ".docx":
        return read_script_from_docx(uploaded_script_path, slide_count)
    elif ext == ".txt":
        return read_script_from_txt(uploaded_script_path, slide_count)
    else:
        raise ValueError(f"不支援的講稿檔案格式：{ext}（僅支援 .docx 或 .txt）")


# ------------------------------------------------------------
# 儲存講稿成 Word 檔（給使用者下載，也方便配音前最後校對）
# ------------------------------------------------------------
def save_scripts_to_docx(slides_text: list, scripts: list, output_path: str):
    """
    將「每頁講稿」整理成一份 Word 文件，存到 output_path。
    對應【輸出】需求中的 講稿.docx。
    """
    doc = Document()
    doc.add_heading("課程講稿", level=1)

    for s, script in zip(slides_text, scripts):
        doc.add_heading(f"第 {s['slide_index']} 頁：{s['title'] or ''}", level=2)
        doc.add_paragraph(script.strip() if script and script.strip() else "（尚無講稿內容）")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    doc.save(output_path)
    log(f"✅ 講稿已儲存為 Word 檔：{output_path}")
    return output_path


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    測試模式一（自己的講稿）與模式三（Speaker Notes），
    這兩種模式不需要呼叫外部 AI API，可以直接離線測試。
    """
    from ppt_reader import load_presentation
    from pptx import Presentation

    test_dir = "/home/claude/PPT2Course_AI"
    test_ppt_path = os.path.join(test_dir, "PPT", "_test_script_gen.pptx")
    test_image_dir = os.path.join(test_dir, "Images", "_test_script_gen")

    # 建立 3 頁測試 PPT（含備忘稿）
    prs = Presentation()
    layout = prs.slide_layouts[1]
    sample_slides = [
        ("課程介紹", "本課程將說明公司資訊安全政策的重要性。", "各位同仁大家好，歡迎參加本次教育訓練課程。"),
        ("資安法規概述", "個人資料保護法\n資通安全管理法", "這一頁我們來看看重要的法規。"),
        ("結語與Q&A", "感謝大家的參與，如有問題歡迎提出。", "課程到此結束，謝謝大家。"),
    ]
    for title, body, notes in sample_slides:
        slide = prs.slides.add_slide(layout)
        slide.shapes.title.text = title
        slide.placeholders[1].text = body
        slide.notes_slide.notes_text_frame.text = notes
    prs.save(test_ppt_path)

    result = load_presentation(test_ppt_path, test_image_dir)
    slides_text = result["slides_text"]

    print("\n===== 測試模式三：Speaker Notes =====")
    notes_scripts = generate_all_scripts("notes", slides_text)
    for s in notes_scripts:
        print(repr(s))

    # 建立測試用講稿 docx（模式一）
    test_script_docx = os.path.join(test_dir, "Script", "_test_user_script.docx")
    doc = Document()
    doc.add_paragraph("第1頁")
    doc.add_paragraph("大家好，這是第一頁的自訂講稿內容，用來測試模式一的頁碼對應功能。")
    doc.add_paragraph("第2頁")
    doc.add_paragraph("這是第二頁的講稿，介紹個資法與資安法。")
    doc.add_paragraph("第3頁")
    doc.add_paragraph("最後一頁，感謝聆聽，歡迎提問。")
    doc.save(test_script_docx)

    print("\n===== 測試模式一：使用者 docx 講稿 =====")
    own_scripts = generate_all_scripts("own", slides_text, uploaded_script_path=test_script_docx)
    for s in own_scripts:
        print(repr(s))

    print("\n===== 測試講稿預覽格式化 / 反解析 =====")
    display_text = format_scripts_for_display(slides_text, own_scripts)
    print(display_text)
    reparsed = parse_scripts_from_display(display_text, len(slides_text))
    assert reparsed == own_scripts, "反解析結果與原始講稿不一致！"
    print("\n✅ 格式化與反解析一致性測試通過")

    print("\n===== 測試儲存為 Word =====")
    output_docx = os.path.join(test_dir, "Output", "_test_講稿.docx")
    save_scripts_to_docx(slides_text, own_scripts, output_docx)
    print("存在:", os.path.exists(output_docx))

    # --- 測試參考資料讀取功能 (PDF / docx / txt) ---
    print("\n===== 測試參考資料讀取 (extract_reference_text) =====")
    from reportlab.pdfgen import canvas as rl_canvas

    ref_pdf_path = os.path.join(test_dir, "Assets", "_test_ref.pdf")
    c = rl_canvas.Canvas(ref_pdf_path)
    c.drawString(100, 800, "Reference PDF Test Content 12345")
    c.save()

    ref_docx_path = os.path.join(test_dir, "Assets", "_test_ref.docx")
    ref_doc = Document()
    ref_doc.add_paragraph("這是參考資料docx測試內容，關鍵字：資安法規測試999")
    ref_doc.save(ref_docx_path)

    ref_txt_path = os.path.join(test_dir, "Assets", "_test_ref.txt")
    with open(ref_txt_path, "w", encoding="utf-8") as f:
        f.write("這是txt參考資料測試內容，關鍵字：個資保護測試888")

    combined = extract_reference_text([ref_pdf_path, ref_docx_path, ref_txt_path])
    print(combined)

    assert "12345" in combined, "PDF 內容擷取失敗"
    assert "999" in combined, "docx 內容擷取失敗"
    assert "888" in combined, "txt 內容擷取失敗"
    print("\n✅ 參考資料讀取（PDF/docx/txt）測試通過")

    # 清理參考資料測試檔案
    for p in [ref_pdf_path, ref_docx_path, ref_txt_path]:
        os.remove(p)


Writing script_generator.py


In [7]:
%%writefile tts.py
# -*- coding: utf-8 -*-
"""
tts.py
======
負責「AI 配音」功能，使用 Edge-TTS（免費、無需 API Key）將講稿文字轉換成語音檔案(mp3)。

主要功能：
1. 提供一份精選的語音清單（繁中為主，也支援簡中/英文），給網頁下拉選單使用
2. 將單一段文字合成為 mp3 音檔（可調整語速、音量）
3. 批次處理整份講稿（每頁一個音檔），並偵測每個音檔的實際時長，
   供後續 video.py 依照配音長度，決定每一頁投影片要停留多久
4. 若某頁沒有講稿內容，會自動產生一小段靜音音檔，確保每一頁都有對應的音檔，
   影片合成階段才不會因為缺檔而出錯
"""

import os
import asyncio
import subprocess

from utils import log
from script_cleaner import clean_script


# ------------------------------------------------------------
# 精選語音清單：畫面上顯示的中文說明 <-> edge-tts 實際的 voice 代碼
# ------------------------------------------------------------
VOICE_OPTIONS = {
    "繁體中文 - 曉臻 (女聲)": "zh-TW-HsiaoChenNeural",
    "繁體中文 - 雲哲 (男聲)": "zh-TW-YunJheNeural",
    "繁體中文 - 曉雨 (女聲)": "zh-TW-HsiaoYuNeural",
    "簡體中文 - 曉曉 (女聲)": "zh-CN-XiaoxiaoNeural",
    "簡體中文 - 雲希 (男聲)": "zh-CN-YunxiNeural",
    "英文(美) - Jenny (女聲)": "en-US-JennyNeural",
    "英文(美) - Guy (男聲)": "en-US-GuyNeural",
}

DEFAULT_VOICE_LABEL = "繁體中文 - 曉臻 (女聲)"


async def list_all_voices() -> list:
    """
    透過 edge-tts 線上查詢「目前所有」可用的語音清單（需要網路）。
    這是給進階使用者或未來擴充語言選單用的輔助函式；
    一般情況下，畫面直接使用上面 VOICE_OPTIONS 精選清單即可，不需要每次都連線查詢。
    """
    import edge_tts

    return await edge_tts.list_voices()


# ------------------------------------------------------------
# 核心：單一段文字轉語音
# ------------------------------------------------------------
async def _synthesize_async(text: str, output_path: str, voice: str, rate: str, volume: str):
    """
    使用 edge-tts 的非同步 API，把文字合成語音並存成 mp3。
    rate / volume 需要是 edge-tts 慣用的字串格式，例如 "+10%"、"-20%"、"+0%"。

    這是「不含精確逐字時間戳記」的簡單版本，作為 _synthesize_with_cues_async() 失敗時的備援。
    """
    import edge_tts

    communicate = edge_tts.Communicate(text, voice=voice, rate=rate, volume=volume)
    await communicate.save(output_path)


async def _synthesize_with_cues_async(text: str, output_path: str, voice: str, rate: str, volume: str) -> str:
    """
    使用 edge-tts 的串流 API，一邊寫入音檔，一邊收集 WordBoundary / SentenceBoundary
    事件（TTS 引擎回報的「每個字真正的發音時間點」），藉此取得精確的字幕時間戳記。

    這是解決「字幕跟配音對不齊」的關鍵：不再用文字長度比例去估算每句話要顯示多久，
    而是直接使用 TTS 引擎本身告訴我們的真實時間點。

    回傳:
        SRT 格式的文字（時間是「相對於這一頁音檔開頭」，從 00:00:00 起算），
        供 subtitle.py 解析後，再依照這一頁在整部影片中的累積時間點做偏移。
        如果這個版本的 edge-tts 不支援 SubMaker 或串流過程出錯，會直接拋出例外，
        由呼叫端 (generate_audio_for_all_slides) 決定要不要 fallback。
    """
    import edge_tts
    from edge_tts import SubMaker

    communicate = edge_tts.Communicate(text, voice=voice, rate=rate, volume=volume)
    submaker = SubMaker()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "wb") as audio_file:
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                audio_file.write(chunk["data"])
            elif chunk["type"] in ("WordBoundary", "SentenceBoundary"):
                submaker.feed(chunk)

    return submaker.get_srt()


def synthesize_speech(
    text: str,
    output_path: str,
    voice: str = "zh-TW-HsiaoChenNeural",
    rate_percent: int = 0,
    volume_percent: int = 0,
) -> str:
    """
    同步包裝函式：把一段文字轉成語音檔（mp3）。
    Gradio 的按鈕事件函式是同步的，這裡用 asyncio.run() 包裝 edge-tts 的非同步呼叫，
    讓上層程式不需要處理 async/await。

    參數:
        text: 要合成的講稿文字（不可為空字串）
        output_path: 輸出的 mp3 檔案路徑
        voice: edge-tts 的語音代碼（可從 VOICE_OPTIONS 的 value 選取）
        rate_percent: 語速調整百分比，例如 20 代表加快20%，-10 代表放慢10%
        volume_percent: 音量調整百分比，例如 20 代表加大20%

    回傳:
        output_path（方便串接呼叫）
    """
    if not text or not text.strip():
        raise ValueError("要合成的文字是空的，請確認講稿內容")

    rate_str = f"{'+' if rate_percent >= 0 else ''}{rate_percent}%"
    volume_str = f"{'+' if volume_percent >= 0 else ''}{volume_percent}%"

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    asyncio.run(_synthesize_async(text, output_path, voice, rate_str, volume_str))
    return output_path


def synthesize_speech_with_cues(
    text: str,
    output_path: str,
    voice: str = "zh-TW-HsiaoChenNeural",
    rate_percent: int = 0,
    volume_percent: int = 0,
) -> str:
    """
    同步包裝函式：把一段文字轉成語音檔，「同時」取得精確的逐字時間戳記(SRT格式字串)。

    這是產生字幕時應該優先使用的版本；只有在這個版本失敗時
    （例如 edge-tts 版本太舊沒有 SubMaker），才 fallback 使用 synthesize_speech()。

    參數同 synthesize_speech()。

    回傳:
        SRT 格式的字幕文字（時間是相對於「這一頁音檔開頭」，從 00:00:00 起算）。
        注意：這裡回傳的是原始、逐字/逐句的精細時間戳記，還沒有合併成適合閱讀的
        字幕段落，也還沒有依照這一頁在整部影片中的累積時間點做偏移——
        這兩步是 subtitle.py 的工作。
    """
    if not text or not text.strip():
        raise ValueError("要合成的文字是空的，請確認講稿內容")

    rate_str = f"{'+' if rate_percent >= 0 else ''}{rate_percent}%"
    volume_str = f"{'+' if volume_percent >= 0 else ''}{volume_percent}%"

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    return asyncio.run(_synthesize_with_cues_async(text, output_path, voice, rate_str, volume_str))


# ------------------------------------------------------------
# 輔助：產生靜音音檔 / 偵測音檔時長 (使用 ffmpeg / ffprobe)
# ------------------------------------------------------------
def generate_silent_audio(output_path: str, duration_seconds: float = 1.5):
    """
    產生一段指定秒數的靜音 mp3 音檔。
    用途：
        1. 某頁投影片沒有講稿內容時的備援音檔（確保每頁都有音檔可用）
        2. AI 配音失敗時的備援，避免整個流程中斷
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-f", "lavfi",
            "-i", "anullsrc=r=24000:cl=mono",
            "-t", str(duration_seconds),
            "-q:a", "9",
            output_path,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )


def get_audio_duration(audio_path: str) -> float:
    """
    使用 ffprobe 偵測音檔的實際長度（秒），回傳浮點數。
    偵測失敗時回傳 0.0，並記錄警告（例如檔案損毀或不存在）。
    """
    if not os.path.exists(audio_path):
        log(f"⚠️ 找不到音檔，無法偵測時長：{audio_path}", "warning")
        return 0.0

    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            audio_path,
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    try:
        return round(float(result.stdout.strip()), 2)
    except (ValueError, TypeError):
        log(f"⚠️ 無法解析音檔時長：{audio_path}", "warning")
        return 0.0


# ------------------------------------------------------------
# 批次：把整份講稿的每一頁轉成語音
# ------------------------------------------------------------
def generate_audio_for_all_slides(
    scripts: list,
    output_dir: str,
    voice: str = "zh-TW-HsiaoChenNeural",
    rate_percent: int = 0,
    volume_percent: int = 0,
    silence_duration_for_empty: float = 1.5,
    progress_callback=None,
) -> list:
    """
    批次把「每頁講稿」轉成語音檔，回傳每頁的音檔資訊清單。

    參數:
        scripts: 每頁講稿文字的清單 (script_generator.generate_all_scripts 的回傳結果)
        output_dir: 音檔輸出資料夾
        voice: edge-tts 語音代碼
        rate_percent / volume_percent: 語速/音量調整百分比
        silence_duration_for_empty: 該頁沒有講稿時，補上的靜音秒數
        progress_callback: 選填，function(fraction: float)，回報進度給 Gradio 進度條

    處理流程（對應每一頁）:
        原始講稿文字
            → script_cleaner.clean_script() 清理排版雜訊（空白行、底線、分隔線等，
              避免 TTS 把這些符號唸出來）
            → 優先呼叫 synthesize_speech_with_cues()，取得音檔 + 真實逐字時間戳記
            → 若失敗，fallback 呼叫 synthesize_speech()，只取得音檔（無精確時間戳記，
              後續 subtitle.py 會改用「依實際音檔時長比例分配」的方式產生字幕，
              而不是憑空估算固定秒數）
            → 一律再用 ffprobe (get_audio_duration) 實際量測音檔時長，
              作為「這一頁要停留多久」的唯一依據

    回傳:
        [
            {
                "slide_index": 1,
                "audio_path": "Audio/xxx/slide_001.mp3",
                "duration": 12.34,          # 秒，來自 ffprobe 實際量測，絕非估算
                "has_audio": True,          # False 代表這頁是靜音(無講稿或配音失敗)
                "srt_cues_raw": "1\\n00:00:00,000 --> ...",  # 精確逐字時間戳記(SRT字串)，
                                                              # 沒有精確時間戳記時為 None
            },
            ...
        ]
    """
    os.makedirs(output_dir, exist_ok=True)
    total = len(scripts)
    results = []

    for i, text in enumerate(scripts):
        slide_num = i + 1
        output_path = os.path.join(output_dir, f"slide_{slide_num:03d}.mp3")

        # 送進 TTS 之前，先清理掉空白行、底線、分隔線等不該被朗讀出來的排版符號
        text = clean_script(text)
        has_audio = False
        srt_cues_raw = None

        if text:
            log(f"配音生成中：第 {slide_num}/{total} 頁")
            try:
                # 優先使用「含精確逐字時間戳記」的版本，字幕才能跟配音完全對齊
                srt_cues_raw = synthesize_speech_with_cues(
                    text, output_path, voice=voice,
                    rate_percent=rate_percent, volume_percent=volume_percent,
                )
                has_audio = True
            except Exception as e:
                log(f"⚠️ 精確時間戳記合成失敗，改用一般配音方式（第 {slide_num} 頁）：{e}", "warning")
                try:
                    synthesize_speech(
                        text, output_path, voice=voice,
                        rate_percent=rate_percent, volume_percent=volume_percent,
                    )
                    has_audio = True
                except Exception as e2:
                    log(f"❌ 第 {slide_num} 頁配音失敗，改用靜音代替：{e2}", "error")
                    generate_silent_audio(output_path, duration_seconds=silence_duration_for_empty)
        else:
            log(f"⚠️ 第 {slide_num} 頁沒有講稿內容，產生 {silence_duration_for_empty} 秒靜音音檔", "warning")
            generate_silent_audio(output_path, duration_seconds=silence_duration_for_empty)

        # 不論用哪種方式合成，「這一頁要停留多久」一律以 ffprobe 實際量測音檔時長為準，
        # 絕不使用文字長度或字數去估算秒數。
        duration = get_audio_duration(output_path)
        results.append({
            "slide_index": slide_num,
            "audio_path": output_path,
            "duration": duration,
            "has_audio": has_audio,
            "srt_cues_raw": srt_cues_raw,
        })

        if progress_callback:
            progress_callback((i + 1) / total)

    total_duration = sum(r["duration"] for r in results)
    log(f"✅ 配音生成完成，共 {total} 個音檔，總長度約 {total_duration:.1f} 秒")
    return results


def format_audio_summary(audio_results: list) -> str:
    """
    把批次配音結果整理成一份 Markdown 表格文字，方便在 Gradio 介面上顯示總覽。
    """
    lines = ["| 頁碼 | 時長(秒) | 狀態 |", "|---|---|---|"]
    total = 0.0
    for r in audio_results:
        status = "✅ 有配音" if r["has_audio"] else "⚠️ 靜音（無講稿或生成失敗）"
        lines.append(f"| {r['slide_index']} | {r['duration']:.1f} | {status} |")
        total += r["duration"]

    minutes = total / 60
    lines.append("")
    lines.append(f"**總長度約：{total:.1f} 秒（約 {minutes:.1f} 分鐘）**")
    return "\n".join(lines)


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    測試不需要網路的部分：
    1. 靜音音檔產生 (generate_silent_audio)
    2. 音檔時長偵測 (get_audio_duration)
    3. 批次流程的整體邏輯（含空白講稿 -> 自動補靜音、失敗 -> 自動補靜音的容錯機制）

    注意：實際呼叫 edge-tts 產生「有聲音」的語音，需要網路環境（例如在 Colab 執行），
    此處沙盒環境沒有網路，所以測試中故意放入正常文字的頁面，
    驗證「即使 edge-tts 呼叫失敗，也會自動 fallback 成靜音、不會讓整個流程中斷」。
    """
    test_dir = "/home/claude/PPT2Course_AI/Audio/_test_tts"
    os.makedirs(test_dir, exist_ok=True)

    # --- 測試 1：靜音音檔 + 時長偵測 ---
    silent_path = os.path.join(test_dir, "silent_test.mp3")
    generate_silent_audio(silent_path, duration_seconds=2.0)
    duration = get_audio_duration(silent_path)
    print(f"靜音音檔時長偵測結果：{duration} 秒（預期約 2.0 秒）")
    assert 1.8 <= duration <= 2.3, "靜音音檔時長偵測結果不符預期！"
    print("✅ 靜音音檔產生 + 時長偵測 測試通過")

    # --- 測試 2：批次流程容錯機制 (無網路環境下，AI配音會失敗並自動fallback成靜音) ---
    test_scripts = [
        "這是第一頁的講稿內容，測試配音生成。",
        "",  # 模擬沒有講稿的頁面
        "第三頁講稿內容測試。",
        "__________",  # 模擬「整行只有底線」的雜訊，清理後應視為空白 -> 產生靜音
    ]
    results = generate_audio_for_all_slides(test_scripts, test_dir, voice="zh-TW-HsiaoChenNeural")

    print("\n批次配音結果：")
    for r in results:
        print(r)

    assert len(results) == 4, "批次配音結果數量不符！"
    assert all(os.path.exists(r["audio_path"]) for r in results), "有音檔沒有被正確產生！"
    assert results[1]["has_audio"] is False, "空白講稿那頁應該標記為 has_audio=False！"
    assert results[3]["has_audio"] is False, "只有底線的講稿，清理後應視為空白 -> has_audio=False！"
    assert all("srt_cues_raw" in r for r in results), "回傳結果應包含 srt_cues_raw 欄位！"
    print("\n✅ 批次配音流程（含容錯機制 + script_cleaner整合）測試通過")

    print("\n===== 配音總覽 (Markdown) =====")
    print(format_audio_summary(results))

    # 清理測試檔案
    import shutil
    shutil.rmtree(test_dir)


Writing tts.py


In [8]:
%%writefile subtitle.py
# -*- coding: utf-8 -*-
"""
subtitle.py
===========
負責「字幕產生」功能：把每頁的講稿文字，依照該頁配音的實際時長，
切割成適合閱讀的字幕段落，並產生時間軸對齊的 SRT 字幕檔。

核心邏輯：
1. 每頁講稿文字先依中文標點（。！？，、）斷句，避免單一字幕行過長看不完
2. 每個字幕段落依「字數比例」分配到該頁配音的時長中
   （沒有真正的語音辨識可以抓每個字的精確發音時間點，
    但用字數比例分配，實務上已經足夠貼近口語講稿的自然停頓）
3. 每頁的時間軸會依序累加（因為最終影片是每頁配音接續播放），
   確保字幕時間點與整部影片的音軌完全對齊
4. 使用 `srt` 套件輸出標準 SRT 格式，任何影片播放器/剪輯軟體都能讀取
"""

import os
import re

from utils import log
from script_cleaner import clean_script


# ------------------------------------------------------------
# SRT 格式輔助函式（純 Python 實作，不依賴外部套件）
# ------------------------------------------------------------
def _format_srt_timestamp(seconds: float) -> str:
    """
    把秒數 (float) 轉換成 SRT 標準時間格式："HH:MM:SS,mmm"
    例如 65.5 秒 -> "00:01:05,500"
    """
    if seconds < 0:
        seconds = 0
    total_ms = int(round(seconds * 1000))
    hours, remainder_ms = divmod(total_ms, 3600000)
    minutes, remainder_ms = divmod(remainder_ms, 60000)
    secs, ms = divmod(remainder_ms, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"


def _compose_srt(entries: list) -> str:
    """
    把字幕段落清單組合成標準 SRT 格式的完整文字內容。

    參數:
        entries: [(index, start_seconds, end_seconds, text), ...]

    回傳:
        SRT 格式字串，每個段落之間以空白行分隔，例如：
            1
            00:00:00,000 --> 00:00:03,500
            各位同仁大家好

            2
            00:00:03,500 --> 00:00:06,000
            歡迎參加本次教育訓練課程
    """
    blocks = []
    for index, start, end, text in entries:
        block = f"{index}\n{_format_srt_timestamp(start)} --> {_format_srt_timestamp(end)}\n{text}"
        blocks.append(block)
    return "\n\n".join(blocks) + "\n"


def parse_srt(srt_content: str) -> list:
    """
    把 SRT 文字內容解析回結構化資料。

    這裡故意採用「先依空白行切割區塊、再從區塊內用正則表達式找時間碼行」的方式，
    而不是用複雜的單一正則表達式一次解析整份內容。原因：
        我們除了解析自己產生的 SRT (_compose_srt) 之外，也需要解析
        edge-tts 的 SubMaker.get_srt() 產生的原始逐字時間戳記字串——
        不同來源的 SRT 在細節格式上可能有些微差異（例如空行數量、
        索引數字是否存在），用區塊切割的方式解析更能容忍這些差異，
        避免因為格式些微不同就解析失敗或錯位。

    回傳:
        [{"index": 1, "start": 0.0, "end": 3.5, "text": "..."}, ...]
    """
    if not srt_content or not srt_content.strip():
        return []

    time_pattern = re.compile(
        r"(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})[,.](\d{3})"
    )

    # 依「一行以上的空白行」切開每個字幕區塊，比用 lookahead 判斷下一區塊起點更穩健，
    # 不會因為不同 SRT 來源的空行數量、換行風格些微差異就解析錯誤。
    blocks = re.split(r"\n\s*\n", srt_content.strip())

    results = []
    for block in blocks:
        lines = [l for l in block.split("\n") if l.strip() != ""]
        if not lines:
            continue

        time_line_idx = None
        match = None
        for i, line in enumerate(lines):
            m = time_pattern.search(line)
            if m:
                time_line_idx = i
                match = m
                break

        if match is None:
            continue  # 這個區塊沒有找到時間碼，略過（例如格式異常的殘留區塊）

        start = (
            int(match.group(1)) * 3600
            + int(match.group(2)) * 60
            + int(match.group(3))
            + int(match.group(4)) / 1000
        )
        end = (
            int(match.group(5)) * 3600
            + int(match.group(6)) * 60
            + int(match.group(7))
            + int(match.group(8)) / 1000
        )

        # 時間碼行之後的所有行都算是字幕文字（有些字幕文字本身會跨多行）
        text_lines = lines[time_line_idx + 1:]
        text = " ".join(t.strip() for t in text_lines).strip()

        # 索引：時間碼行前面若有一行純數字，當作 index；否則依解析順序自動編號
        idx = len(results) + 1
        if time_line_idx > 0 and lines[time_line_idx - 1].strip().isdigit():
            idx = int(lines[time_line_idx - 1].strip())

        results.append({"index": idx, "start": start, "end": end, "text": text})

    return results


# ------------------------------------------------------------
# 步驟 1：把一段講稿文字切割成適合當字幕的小段落
# ------------------------------------------------------------
# 句尾標點：優先在這些符號後面斷句，語意較完整
_SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[。！？；：!?;:])")
# 次要斷點：句子太長時，優先在這些符號後面找地方換行/分段
_CLAUSE_SPLIT_PATTERN = re.compile(r"(?<=[，、,])")

# 單行字幕建議字數區間（中文字），對應「15~25個中文字」的需求
SUBTITLE_MIN_LINE_CHARS = 15
SUBTITLE_MAX_LINE_CHARS = 25
# 可斷行的字元（標點、空白），優先在這些字元「之後」換行，避免斷在詞彙中間
_LINE_BREAK_AFTER_CHARS = set("，、,　 」』】）")


def wrap_subtitle_text(text: str, max_line_chars: int = SUBTITLE_MAX_LINE_CHARS) -> str:
    """
    把一段字幕文字，依照「每行約 15~25 個中文字、最多兩行」的原則自動換行，
    避免字幕整段擠在一行、超出畫面寬度。

    換行策略：
        1. 文字長度在 max_line_chars 以內：不換行，單行顯示
        2. 超過的話：從文字中點附近，優先找標點/空白處換行（避免斷在詞彙中間）；
           找不到適合的斷點才直接在字數上限處硬斷
        3. 最多只切成兩行（若文字真的非常長，第二行會包含剩餘全部內容，
           不會無限往下多切，避免字幕佔滿畫面）

    參數:
        text: 單一字幕段落的文字（已經是要顯示在同一個時間區間的完整內容）
        max_line_chars: 單行字數上限

    回傳:
        已插入換行符號（如果需要）的字幕文字
    """
    text = (text or "").strip()
    if len(text) <= max_line_chars:
        return text

    mid = len(text) // 2
    best_pos = None
    # 從中點往左右兩側搜尋最接近的「可斷行字元」位置
    for offset in range(0, len(text)):
        for pos in (mid + offset, mid - offset):
            if 0 < pos < len(text) and text[pos - 1] in _LINE_BREAK_AFTER_CHARS:
                best_pos = pos
                break
        if best_pos:
            break

    if best_pos is None:
        # 找不到適合的標點/空白斷點，保底直接在字數上限處硬斷
        best_pos = min(max_line_chars, len(text) - 1)

    line1 = text[:best_pos].strip()
    line2 = text[best_pos:].strip()
    return f"{line1}\n{line2}" if line2 else line1


def split_text_into_chunks(text: str, max_chars: int = 20) -> list:
    """
    將一段講稿文字，切割成適合當作「單一字幕段落」顯示的文字清單。

    切割策略：
        1. 優先依照句尾標點（。！？；：）斷句，這通常對應語意完整的一句話
        2. 若單一句子仍然超過 max_chars，再依逗號/頓號進一步切割並重新組合，
           確保每段字幕以「單行」顯示時長度適中、不會太長看不完
           （中文字幕常見建議一行不超過 30~40 字）
        3. 若完全沒有標點（例如AI生成的講稿偶爾會這樣），則依長度與空白字元
           自動切句，避免整段文字擠成一行

    參數:
        text: 該頁的講稿文字
        max_chars: 單一字幕段落（單行）的建議最大字數

    回傳:
        字幕段落文字的 list，依照原文順序排列
    """
    text = (text or "").strip()
    if not text:
        return []

    sentences = [s.strip() for s in _SENTENCE_SPLIT_PATTERN.split(text) if s.strip()]
    if not sentences:
        sentences = [text]

    chunks = []
    for sentence in sentences:
        if len(sentence) <= max_chars:
            chunks.append(sentence)
            continue

        # 句子太長，先依逗號/頓號切成更小的片段，再重新組合成不超過 max_chars 的段落
        clauses = [c.strip() for c in _CLAUSE_SPLIT_PATTERN.split(sentence) if c.strip()]
        if len(clauses) == 1:
            # 完全沒有逗號/頓號可切（沒有標點的長句），改用空白或長度自動切句
            words = sentence.split(" ") if " " in sentence else None
            if words and len(words) > 1:
                clauses = []
                buf = ""
                for w in words:
                    if len(buf) + len(w) + 1 <= max_chars:
                        buf = f"{buf} {w}".strip()
                    else:
                        if buf:
                            clauses.append(buf)
                        buf = w
                if buf:
                    clauses.append(buf)
            else:
                # 連空白都沒有，直接依字數長度切
                clauses = [sentence[i:i + max_chars] for i in range(0, len(sentence), max_chars)]

        buffer = ""
        for clause in clauses:
            if len(buffer) + len(clause) <= max_chars:
                buffer += clause
            else:
                if buffer:
                    chunks.append(buffer)
                # 極端情況：單一片段本身就超過 max_chars（沒有逗號可切），直接強制切斷
                if len(clause) > max_chars:
                    for i in range(0, len(clause), max_chars):
                        chunks.append(clause[i: i + max_chars])
                    buffer = ""
                else:
                    buffer = clause
        if buffer:
            chunks.append(buffer)

    return chunks



# ------------------------------------------------------------
# 步驟 2：依字數比例，把時長分配給每個字幕段落
# ------------------------------------------------------------
def allocate_chunk_timings(chunks: list, total_duration: float, min_display_seconds: float = 1.0) -> list:
    """
    依照每個字幕段落的字數比例，分配顯示時間；並確保每段至少顯示
    min_display_seconds 秒，避免字數很少的段落一閃而過看不清楚。

    參數:
        chunks: split_text_into_chunks() 產生的字幕段落清單
        total_duration: 這一頁配音的總時長（秒）
        min_display_seconds: 每段字幕最少顯示秒數

    回傳:
        [(字幕文字, 顯示秒數), ...]，所有顯示秒數加總會等於 total_duration
    """
    if not chunks or total_duration <= 0:
        return []

    if len(chunks) == 1:
        return [(chunks[0], total_duration)]

    total_chars = sum(len(c) for c in chunks) or 1
    raw_durations = [
        max(min_display_seconds, total_duration * (len(c) / total_chars)) for c in chunks
    ]

    # 套用最低顯示秒數後，總和可能超過 total_duration，
    # 這裡按比例縮放回去，確保整頁字幕時間軸仍然精準對齊音檔長度。
    raw_total = sum(raw_durations)
    scale = total_duration / raw_total if raw_total > 0 else 1.0
    final_durations = [d * scale for d in raw_durations]

    return list(zip(chunks, final_durations))


# ------------------------------------------------------------
# 步驟 2b：把 TTS 引擎回報的「真實逐字時間戳記」合併成適合閱讀的字幕段落
# ------------------------------------------------------------
_SENTENCE_END_CHARS = set("。！？；：!?;:")


def merge_cues_to_readable_chunks(raw_cues: list, max_chars: int = 20) -> list:
    """
    把 tts.py 用 edge-tts WordBoundary/SentenceBoundary 事件產生的「逐字/逐句」
    原始字幕段落，合併成適合閱讀的字幕段落——但合併時，起始/結束時間一律直接沿用
    這些原始段落「真實量測到」的時間點，絕對不會重新估算，確保字幕跟配音100%對齊。

    合併規則：
        - 持續合併相鄰段落，直到累積字數達到 max_chars
        - 若遇到句尾標點（。！？；：），優先在該處斷開（語意較完整），
          即使還沒到 max_chars 也會斷開
        - 若加入下一段會讓目前緩衝超過 max_chars，會先把目前緩衝收尾，
          再從這一段重新開始累積
        - 【重要】若「單一原始段落本身」就已經超過 max_chars
          （常見於 edge-tts 的 SentenceBoundary 事件，一次給的就是一整句完整長句，
          而不是逐字），會在這一段「真實量測到」的時間範圍內，依文字比例
          進一步切割成多個較短的段落 —— 這是先前版本遺漏的情況，沒有處理的話，
          長句會整句原封不動變成一行字幕，超出畫面邊界。因為子切割沒有更細的
          逐字時間可用，才會在這個「已經是真實量測」的小範圍內用比例估算，
          誤差侷限在單一句子的時間長度內，非常小。
        - 合併後的段落：開始時間 = 這一段第一個原始段落的開始時間，
                        結束時間 = 這一段最後一個原始段落的結束時間

    參數:
        raw_cues: [{"start": float, "end": float, "text": str}, ...]，
                  由 subtitle.parse_srt() 解析 tts.py 回傳的原始 SRT 字串而來，
                  時間是「相對於這一頁音檔開頭」（從 0 開始）
        max_chars: 單一字幕段落的建議最大字數

    回傳:
        [{"start": float, "end": float, "text": str}, ...]，合併後的字幕段落，
        時間同樣是「相對於這一頁音檔開頭」，尚未做累積偏移
    """
    if not raw_cues:
        return []

    merged = []
    buffer_texts = []
    buffer_start = None
    prev_end = raw_cues[0]["start"]

    def _split_oversized_buffer(text: str, start_time: float, end_time: float):
        """把單一段落內部（本身已超過 max_chars）依文字比例切割成多個較短段落。"""
        sub_chunks = split_text_into_chunks(text, max_chars=max_chars)
        total_dur = max(end_time - start_time, 0.001)
        total_chars = sum(len(c) for c in sub_chunks) or 1
        t = start_time
        for sc in sub_chunks:
            sc_dur = total_dur * (len(sc) / total_chars)
            merged.append({"start": t, "end": t + sc_dur, "text": sc})
            t += sc_dur

    for cue in raw_cues:
        cue_text = cue["text"]
        combined_len = sum(len(t) for t in buffer_texts) + len(cue_text)

        # 加入這一段會讓目前緩衝超過上限：先把目前已累積的緩衝收尾，
        # 這一段重新開始累積（而不是硬塞進去讓緩衝爆掉）
        if buffer_texts and combined_len > max_chars:
            buffer_text = "".join(buffer_texts)
            if len(buffer_text) > max_chars:
                _split_oversized_buffer(buffer_text, buffer_start, prev_end)
            else:
                merged.append({"start": buffer_start, "end": prev_end, "text": buffer_text})
            buffer_texts = []
            buffer_start = None

        if buffer_start is None:
            buffer_start = cue["start"]
        buffer_texts.append(cue_text)
        prev_end = cue["end"]

        current_text = "".join(buffer_texts)
        ends_sentence = cue_text.strip()[-1:] in _SENTENCE_END_CHARS if cue_text.strip() else False

        if len(current_text) > max_chars:
            # 單一段落本身就超過上限（例如 SentenceBoundary 直接給出一整句長句），
            # 在這段真實量測到的時間範圍內，依文字比例進一步切割。
            _split_oversized_buffer(current_text, buffer_start, prev_end)
            buffer_texts = []
            buffer_start = None
        elif len(current_text) >= max_chars or ends_sentence:
            merged.append({"start": buffer_start, "end": prev_end, "text": current_text})
            buffer_texts = []
            buffer_start = None

    # 收尾：如果還有殘留、尚未組成一段的文字，補上最後一段
    if buffer_texts:
        buffer_text = "".join(buffer_texts)
        if len(buffer_text) > max_chars:
            _split_oversized_buffer(buffer_text, buffer_start, prev_end)
        else:
            merged.append({
                "start": buffer_start,
                "end": prev_end,
                "text": buffer_text,
            })

    return merged


# ------------------------------------------------------------
# 步驟 3：整合成整份課程的 SRT 字幕
# ------------------------------------------------------------
def generate_full_srt(
    scripts: list,
    audio_results: list,
    max_chars_per_line: int = 20,
    transition_duration: float = 0.0,
) -> str:
    """
    依照「每頁講稿文字」+「每頁音檔實際時長」，產生整份課程的 SRT 字幕內容。

    時間來源優先順序（絕不使用固定秒數或文字長度去「憑空估算」）：
        1. 【優先】如果 audio_results[i]["srt_cues_raw"] 有值
           （tts.py 用 edge-tts 的 WordBoundary/SentenceBoundary 事件產生，
           是 TTS 引擎回報的「真實」逐字發音時間點），
           直接採用這些真實時間戳記合併成字幕段落，只是合併多個小段落成適合閱讀的
           長度，時間邊界本身不做任何估算或改動。
        2. 【備援】如果沒有真實時間戳記（例如該頁配音是舊流程產生、或 edge-tts
           版本不支援 SubMaker），才 fallback 改用「依文字字數比例分配」的方式，
           並且仍然是分配在「該頁 ffprobe 實際量測出來的音檔時長」之內，
           不是憑空指定固定幾秒。

    每一頁在整部影片時間軸上的「起訖點」，用 audio_results[i]["duration"]
    （ffprobe 實際量測的音檔時長）+ transition_duration（轉場時間）來累加。
    這裡的 transition_duration 必須跟 video.py 的
    build_slides_with_transitions() 使用「同一個數值」，
    因為那邊每一頁片段在配音結束後，都會多留 transition_duration 秒的
    轉場緩衝時間才切到下一頁 —— 字幕的時間軸也要把這段緩衝時間算進去，
    否則字幕會比實際畫面/配音提早出現在下一頁（這是三者對齊的關鍵）。

    參數:
        scripts: 每頁講稿文字清單（與投影片頁數一一對應）
        audio_results: tts.generate_audio_for_all_slides() 的回傳結果，
                       每個元素需含 "duration" (秒)；若含 "srt_cues_raw"
                       則會優先採用其中的真實時間戳記
        max_chars_per_line: 單一字幕段落（單行顯示）的建議最大字數
        transition_duration: 影片每頁之間的轉場時間（秒），須與影片合成時
                             使用的數值一致，才能維持字幕/配音/影片三者同步

    回傳:
        SRT 格式的完整字幕文字內容（字串）
    """
    if len(scripts) != len(audio_results):
        log(
            f"⚠️ 講稿頁數({len(scripts)}) 與音檔數量({len(audio_results)}) 不一致，"
            "字幕可能無法完整對齊，請確認是否已重新產生講稿或配音。",
            "warning",
        )

    n = len(audio_results)
    entries = []
    index = 1
    current_offset = 0.0  # 目前累積的時間點(秒)，因為每頁影片會依序播放
    used_real_timestamps_count = 0
    used_fallback_count = 0

    for page_idx, (text, audio_info) in enumerate(zip(scripts, audio_results)):
        # 統一在這裡先清理一次講稿文字（跟 tts.py 送進 TTS 前的清理邏輯是同一個
        # clean_script() 函式）。這一步很重要：即使外部呼叫者（例如 app.py）
        # 傳進來的是「使用者編輯框裡的原始文字」，也要確保字幕呈現的內容
        # 跟「TTS 實際念出來的內容」一致，不會把空白行、底線分隔線等雜訊
        # 顯示成字幕（這是實測時抓到的真實 bug：備援的比例估算路徑如果不清理，
        # 就會把「_________」這種分隔線原封不動地當作一句字幕顯示出來）。
        text = clean_script(text)

        duration = audio_info.get("duration", 0.0) or 0.0
        raw_srt = audio_info.get("srt_cues_raw")
        # 只有「不是最後一頁」才需要加上轉場緩衝時間（跟 video.py 的邏輯一致）
        gap_after = transition_duration if (transition_duration > 0 and page_idx < n - 1) else 0.0

        if duration <= 0:
            current_offset += duration + gap_after
            continue

        page_chunks = None  # 最終會是 [(text, start_within_page, end_within_page), ...]

        if raw_srt:
            try:
                raw_cues = parse_srt(raw_srt)
                if raw_cues:
                    merged = merge_cues_to_readable_chunks(raw_cues, max_chars=max_chars_per_line)
                    if merged:
                        page_chunks = [(m["text"], m["start"], m["end"]) for m in merged]
                        used_real_timestamps_count += 1
            except Exception as e:
                log(f"⚠️ 解析真實時間戳記失敗，改用比例估算方式：{e}", "warning")

        if page_chunks is None:
            # 備援：沒有真實時間戳記時，才依文字字數比例分配（仍限制在實際音檔時長內）
            chunks = split_text_into_chunks(text, max_chars=max_chars_per_line)
            if not chunks:
                current_offset += duration + gap_after
                continue
            timed_chunks = allocate_chunk_timings(chunks, duration)
            page_chunks = []
            t = 0.0
            for chunk_text, chunk_duration in timed_chunks:
                page_chunks.append((chunk_text, t, t + chunk_duration))
                t += chunk_duration
            used_fallback_count += 1
        else:
            # 保險：真實時間戳記可能因為編碼/取樣誤差，跟 ffprobe 量測的檔案長度
            # 有極小的差異(通常 <0.1秒)，這裡把每一段的時間都鉗制在 [0, duration] 範圍內，
            # 避免字幕時間點「跑到下一頁去」。
            page_chunks = [
                (t, max(0.0, min(s, duration)), max(0.0, min(e, duration)))
                for t, s, e in page_chunks
            ]

        for chunk_text, start_within_page, end_within_page in page_chunks:
            entries.append((
                index,
                current_offset + start_within_page,
                current_offset + end_within_page,
                chunk_text,
            ))
            index += 1

        current_offset += duration + gap_after

    srt_content = _compose_srt(entries)
    log(
        f"✅ 字幕產生完成，共 {len(entries)} 段字幕，總長度約 {current_offset:.1f} 秒"
        f"（{used_real_timestamps_count} 頁使用真實時間戳記，"
        f"{used_fallback_count} 頁使用比例估算備援）"
    )
    return srt_content


def save_srt_file(srt_content: str, output_path: str) -> str:
    """把 SRT 字幕內容存成檔案，回傳輸出路徑。"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(srt_content)
    log(f"✅ 字幕已儲存：{output_path}")
    return output_path


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    測試三個層次：
    1. 文字斷句/切割邏輯 (split_text_into_chunks)
    2. 時間分配邏輯是否精準加總回原本的時長 (allocate_chunk_timings)
    3. 整份 SRT 產生 + 時間軸累加是否正確、且能被 srt 套件正確解析回來 (往返一致性測試)
    """
    # --- 測試 1：斷句切割 ---
    sample_text = "各位同仁大家好，歡迎參加本次教育訓練課程。今天我們要來討論職場霸凌的定義、常見樣態，以及公司內部的申訴管道，希望大家都能建立正確的認知。"
    chunks = split_text_into_chunks(sample_text, max_chars=20)
    print("===== 測試 1：文字切割 =====")
    for c in chunks:
        print(f"({len(c)}字) {c}")
    assert all(len(c) <= 25 for c in chunks), "有段落明顯超過長度限制！"  # 留一點緩衝給極端切法
    print("✅ 文字切割測試通過\n")

    # --- 測試 2：時間分配 ---
    print("===== 測試 2：時間分配 =====")
    total_dur = 12.0
    timed = allocate_chunk_timings(chunks, total_dur)
    for text, dur in timed:
        print(f"{dur:.2f}秒 - {text}")
    total_allocated = sum(d for _, d in timed)
    print(f"分配時間總和：{total_allocated:.2f} 秒（預期 {total_dur} 秒）")
    assert abs(total_allocated - total_dur) < 0.01, "時間分配總和與原始時長不符！"
    print("✅ 時間分配測試通過\n")

    # --- 測試 3：整份 SRT 產生 + 時間軸累加 + 往返解析一致性 ---
    print("===== 測試 3：整份 SRT 產生 =====")
    test_scripts = [
        "各位同仁大家好，歡迎參加本次教育訓練課程。",
        "",  # 模擬靜音頁（無講稿）
        "本課程將說明職場霸凌的定義與申訴管道，感謝大家的參與。",
    ]
    test_audio_results = [
        {"slide_index": 1, "duration": 5.0},
        {"slide_index": 2, "duration": 2.0},  # 靜音頁
        {"slide_index": 3, "duration": 8.0},
    ]

    srt_content = generate_full_srt(test_scripts, test_audio_results, max_chars_per_line=20)
    print(srt_content)

    # 往返一致性：重新解析剛剛產生的內容，確認格式正確、可被讀回
    parsed = parse_srt(srt_content)
    print(f"解析回來的字幕段落數：{len(parsed)}")
    assert len(parsed) > 0, "SRT 內容解析失敗或是空的！"

    # 確認第三頁的字幕，時間點有正確累加第一頁(5秒)+第二頁靜音(2秒)之後才開始
    third_slide_start = parsed[-1]["start"]
    print(f"第三頁第一段字幕開始時間：{third_slide_start:.2f} 秒（預期接近 7.0 秒之後）")
    assert third_slide_start >= 7.0 - 0.01, "時間軸累加邏輯有誤，第三頁字幕開始時間不對！"

    # 確認整體結尾時間點，接近 5+2+8 = 15 秒
    last_end = parsed[-1]["end"]
    print(f"最後一段字幕結束時間：{last_end:.2f} 秒（預期接近 15.0 秒）")
    assert abs(last_end - 15.0) < 0.1, "整體時間軸總長度與音檔總長度不符！"

    print("\n✅ 整份 SRT 產生 + 時間軸對齊 + 往返解析一致性 測試通過")

    # --- 測試存檔 ---
    output_path = "/home/claude/PPT2Course_AI/Output/_test_字幕.srt"
    save_srt_file(srt_content, output_path)
    assert os.path.exists(output_path)
    os.remove(output_path)
    print("✅ SRT 存檔測試通過")

    # --- 測試 4：合併真實逐字時間戳記 (merge_cues_to_readable_chunks) ---
    print("\n===== 測試 4：合併真實逐字時間戳記 =====")
    # 模擬 tts.py 從 edge-tts WordBoundary 收集到的逐字時間戳記(每字0.3秒)
    raw_cues = [
        {"start": 0.0, "end": 0.3, "text": "各"},
        {"start": 0.3, "end": 0.6, "text": "位"},
        {"start": 0.6, "end": 0.9, "text": "同"},
        {"start": 0.9, "end": 1.2, "text": "仁"},
        {"start": 1.2, "end": 1.5, "text": "好"},
        {"start": 1.5, "end": 1.8, "text": "。"},
        {"start": 1.8, "end": 2.1, "text": "歡"},
        {"start": 2.1, "end": 2.4, "text": "迎"},
        {"start": 2.4, "end": 2.7, "text": "參"},
        {"start": 2.7, "end": 3.0, "text": "加"},
        {"start": 3.0, "end": 3.3, "text": "。"},
    ]
    merged = merge_cues_to_readable_chunks(raw_cues, max_chars=20)
    print("合併結果：")
    for m in merged:
        print(f"  {m['start']:.1f}s ~ {m['end']:.1f}s : {m['text']}")

    # 應該在句尾標點處斷開，且時間直接沿用原始逐字時間戳記(沒有任何估算)
    assert len(merged) == 2, "應該在句尾標點處斷成2段！"
    assert merged[0]["text"] == "各位同仁好。", "第一段合併文字不正確！"
    assert merged[0]["start"] == 0.0 and merged[0]["end"] == 1.8, "第一段時間應直接沿用原始時間戳記！"
    assert merged[1]["text"] == "歡迎參加。", "第二段合併文字不正確！"
    assert merged[1]["start"] == 1.8 and merged[1]["end"] == 3.3, "第二段時間應直接沿用原始時間戳記！"
    print("✅ 合併真實逐字時間戳記 測試通過（時間完全沿用真實量測值，無任何估算）")

    # --- 測試 5：generate_full_srt 優先使用真實時間戳記 (srt_cues_raw) ---
    print("\n===== 測試 5：generate_full_srt 優先使用真實時間戳記 =====")
    fake_raw_srt_page1 = _compose_srt([
        (1, 0.0, 1.8, "各位同仁好。"),
        (2, 1.8, 3.3, "歡迎參加。"),
    ])
    test_scripts_2 = ["各位同仁好。歡迎參加。", "第二頁沒有真實時間戳記的講稿。"]
    test_audio_results_2 = [
        {"slide_index": 1, "duration": 3.3, "srt_cues_raw": fake_raw_srt_page1},
        {"slide_index": 2, "duration": 4.0, "srt_cues_raw": None},  # 沒有真實時間戳記 -> 應fallback
    ]
    srt2 = generate_full_srt(test_scripts_2, test_audio_results_2, max_chars_per_line=20)
    print(srt2)
    parsed2 = parse_srt(srt2)

    # 第一頁應該直接採用真實時間戳記(0~1.8, 1.8~3.3)，第二頁應該fallback比例估算(從3.3秒開始累加)
    page1_entries = [p for p in parsed2 if p["end"] <= 3.3 + 0.001]
    assert len(page1_entries) == 2, "第一頁應該有2段字幕(來自真實時間戳記)！"
    assert page1_entries[0]["start"] == 0.0 and page1_entries[0]["end"] == 1.8
    assert page1_entries[1]["start"] == 1.8 and page1_entries[1]["end"] == 3.3
    assert all(p["start"] >= 3.3 - 0.001 for p in parsed2 if p not in page1_entries), \
        "第二頁字幕應該從第一頁結束時間(3.3秒)之後才開始！"
    print("✅ generate_full_srt 優先使用真實時間戳記、並正確fallback第二頁 測試通過")

    # --- 測試 6：字幕自動換行 (wrap_subtitle_text) ---
    print("\n===== 測試 6：字幕自動換行 =====")
    short_text = "今天介紹AI"
    assert wrap_subtitle_text(short_text) == short_text, "短文字不應該被換行！"

    long_text = "今天我們要介紹人工智慧的基本概念，並且說明它在日常生活中的實際應用場景"
    wrapped = wrap_subtitle_text(long_text, max_line_chars=20)
    print(f"換行前：{long_text} ({len(long_text)}字)")
    print(f"換行後：\n{wrapped}")
    lines = wrapped.split("\n")
    assert len(lines) == 2, "超過長度限制的文字應該被換成兩行！"
    assert all(len(l) <= 25 for l in lines), "換行後每行不應超過25字太多！"
    print("✅ 字幕自動換行測試通過")

    # --- 測試 7：分號/冒號也能正確斷句 ---
    print("\n===== 測試 7：分號/冒號斷句 =====")
    text_with_colon = "常見樣態包含：言語霸凌；肢體霸凌；關係排擠。"
    chunks7 = split_text_into_chunks(text_with_colon, max_chars=44)
    print("斷句結果：", chunks7)
    assert len(chunks7) >= 1
    print("✅ 分號/冒號斷句測試通過")

    # --- 測試 8：完全沒有標點時，依空白/長度自動斷句 ---
    print("\n===== 測試 8：無標點自動斷句 =====")
    no_punct_text = "今天我們要來介紹一下這個系統的基本操作方式跟注意事項讓大家可以快速上手使用"
    chunks8 = split_text_into_chunks(no_punct_text, max_chars=20)
    print("斷句結果：")
    for c in chunks8:
        print(f"  ({len(c)}字) {c}")
    assert len(chunks8) > 1, "沒有標點的長文字，也應該被自動切成多段！"
    assert all(len(c) <= 25 for c in chunks8), "每段長度應在合理範圍內！"
    print("✅ 無標點自動斷句測試通過")

    # --- 測試 9：generate_full_srt 加入 transition_duration 後，累積時間軸正確納入轉場緩衝 ---
    print("\n===== 測試 9：轉場時間納入字幕累積時間軸 =====")
    test_scripts_3 = ["第一頁講稿。", "第二頁講稿。", "第三頁講稿。"]
    test_audio_results_3 = [
        {"slide_index": 1, "duration": 2.0},
        {"slide_index": 2, "duration": 3.0},
        {"slide_index": 3, "duration": 1.5},
    ]
    D = 0.5
    srt3 = generate_full_srt(test_scripts_3, test_audio_results_3, transition_duration=D)
    parsed3 = parse_srt(srt3)
    print(srt3)

    # 第二頁字幕應該從 (2.0 + D) 開始；第三頁應該從 (2.0+D+3.0+D) 開始
    expected_page2_start = 2.0 + D
    expected_page3_start = 2.0 + D + 3.0 + D
    page2_start = parsed3[1]["start"]
    page3_start = parsed3[2]["start"]
    print(f"第二頁字幕開始時間: {page2_start:.2f}s (預期 {expected_page2_start:.2f}s)")
    print(f"第三頁字幕開始時間: {page3_start:.2f}s (預期 {expected_page3_start:.2f}s)")
    assert abs(page2_start - expected_page2_start) < 0.01, "轉場時間沒有正確納入第二頁的累積時間軸！"
    assert abs(page3_start - expected_page3_start) < 0.01, "轉場時間沒有正確納入第三頁的累積時間軸！"
    print("✅ 轉場時間正確納入字幕累積時間軸 測試通過（跟 video.py 的轉場邏輯一致）")

    # --- 測試 10：確認字幕改回「單行顯示」（比照 v7 版本），不再強制切成兩行 ---
    print("\n===== 測試 10：字幕改回單行顯示（v7 風格） =====")
    long_script = [
        "今天我們要介紹人工智慧的基本概念，並且說明它在日常生活中的實際應用場景，"
        "希望大家可以更加了解這項技術帶來的影響。"
    ]
    long_audio = [{"slide_index": 1, "duration": 10.0}]
    srt10 = generate_full_srt(long_script, long_audio)
    parsed10 = parse_srt(srt10)
    print(srt10)
    for p in parsed10:
        assert "\n" not in p["text"], f"字幕不應該再包含換行符號（應為單行顯示）：{p['text']!r}"
        assert len(p["text"]) <= 35, f"單行字幕不應超過35字：({len(p['text'])}字) {p['text']!r}"
    print(f"✅ 共 {len(parsed10)} 段字幕，皆為單行顯示、每行不超過35字，符合 v7 版本風格")

    # --- 測試 11：真實回報「單一段落本身就超過max_chars」的情境
    #     (例如 edge-tts 用 SentenceBoundary 一次給出一整句長句，而不是逐字) ---
    print("\n===== 測試 11：單一段落過長時的二次切分（本次修正的bug） =====")
    long_sentence_cue = [
        # 模擬 SentenceBoundary：一整句 47 字，只有一個時間區間 [0.0, 4.7]
        {"start": 0.0, "end": 4.7,
         "text": "的基本觀念，包括新制度重點、哪些情況可能屬於職場霸凌、常見霸凌行為，以及申訴管道說明。"},
    ]
    merged11 = merge_cues_to_readable_chunks(long_sentence_cue, max_chars=20)
    print("合併結果：")
    for m in merged11:
        print(f"  {m['start']:.2f}s ~ {m['end']:.2f}s ({len(m['text'])}字): {m['text']}")

    assert len(merged11) > 1, "單一過長段落應該被二次切分成多段，不應該原封不動輸出一整句！"
    for m in merged11:
        assert len(m["text"]) <= 20, f"二次切分後仍有段落超過字數上限：({len(m['text'])}字) {m['text']!r}"
    # 時間軸應該仍然落在原始 cue 的 [0.0, 4.7] 範圍內，且遞增不重疊
    assert abs(merged11[0]["start"] - 0.0) < 0.01
    assert abs(merged11[-1]["end"] - 4.7) < 0.01
    for i in range(1, len(merged11)):
        assert merged11[i]["start"] >= merged11[i - 1]["end"] - 0.001
    print(f"✅ 單一過長段落被正確二次切分成 {len(merged11)} 段，時間軸仍在原始真實區間內、無重疊")

    # 同步驗證 generate_full_srt 整合這種情境時，也不會產生超長字幕
    fake_long_raw_srt = _compose_srt([(1, 0.0, 4.7, long_sentence_cue[0]["text"])])
    test_audio_11 = [{"slide_index": 1, "duration": 4.7, "srt_cues_raw": fake_long_raw_srt}]
    srt11 = generate_full_srt(["(略)"], test_audio_11, max_chars_per_line=20)
    parsed11 = parse_srt(srt11)
    print("\ngenerate_full_srt 整合結果：")
    print(srt11)
    for p in parsed11:
        assert len(p["text"]) <= 20, f"整合後仍出現超長字幕：({len(p['text'])}字) {p['text']!r}"
    print("✅ generate_full_srt 整合後也確認沒有任何超長字幕段落")


Writing subtitle.py


In [9]:
%%writefile video.py
# -*- coding: utf-8 -*-
"""
video.py
========
負責「影片合成」功能：把每頁投影片圖片 + 對應配音，依序組合成一部完整的課程 MP4 影片，
並可選擇加入轉場效果與字幕。

【技術選擇說明】
原始規劃使用 MoviePy + FFmpeg。但 MoviePy 在 2.0 版做了大量破壞性 API 改版
（例如 set_duration -> with_duration、moviepy.editor 整個被移除等），
且此開發環境沒有網路可以安裝 MoviePy 實際測試新版 API 是否正確可用。
考量到穩定性與「每一步都要能真正測試過」的原則，這裡改為直接呼叫系統的
FFmpeg（透過 subprocess），所有濾鏡(filter)組合都已經在開發過程中實際
產生影片並驗證畫面效果，而不是憑印象寫程式碼。FFmpeg 本身沒有版本破壞性
改版的問題，長期維護也更穩定。

主要功能：
1. 把單一投影片圖片 + 配音，合成一小段 MP4 片段，並套用選擇的轉場效果：
   - fade（淡入淡出）
   - zoom（緩慢放大，Ken Burns 效果）
   - slide（從右側滑入）
   - none（無效果，直接顯示）
2. 把所有片段依序串接成完整影片
3. 選擇性把字幕「燒錄」進影片畫面中（硬字幕，任何播放器都看得到）
"""

import os
import subprocess

from utils import log, run_subprocess


# ------------------------------------------------------------
# 共用設定
# ------------------------------------------------------------
DEFAULT_RESOLUTION = (1920, 1080)
DEFAULT_FPS = 30

# 轉場「風格」<-> FFmpeg xfade 濾鏡實際使用的轉場名稱。
# "none" 是特例，代表不使用 xfade，直接卡點切換（不需要額外轉場時間）。
# 其餘皆對應到 FFmpeg xfade 濾鏡「真正支援」的轉場效果（已用 `ffmpeg -h filter=xfade`
# 查證過，非憑印象猜測），是真正的兩張畫面交叉混合效果，不是簡易的淡入淡出特效。
XFADE_TRANSITION_MAP = {
    "none": None,
    "fade": "fadeblack",       # 傳統「淡出到黑，再淡入」的 Fade
    "crossfade": "fade",       # xfade 的 "fade" 本身就是直接交叉淡化(不經過黑)，對應「Cross Fade」
    "dissolve": "dissolve",    # 像素溶解效果
    "slideleft": "slideleft",
    "slideright": "slideright",
    "zoom": "zoomin",
}
TRANSITION_CHOICES = list(XFADE_TRANSITION_MAP.keys())

# 使用者可選擇的轉場時間長度（秒）
TRANSITION_DURATION_CHOICES = [0, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0]
DEFAULT_TRANSITION_DURATION = 0.5

# 字幕燒錄使用的中文字型：系統內建的 Noto Sans CJK，確保繁體中文可以正確顯示
SUBTITLE_FONT_NAME = "Noto Sans CJK TC"


def _fit_filter(width: int, height: int) -> str:
    """
    產生「等比例縮放 + 置中補黑邊」的 FFmpeg 濾鏡字串。
    確保不論投影片原始比例是 4:3 或 16:9，都能完整顯示、不變形，
    多餘的部分用黑邊補滿到目標解析度。
    """
    return (
        f"scale={width}:{height}:force_original_aspect_ratio=decrease,"
        f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color=black"
    )


# ------------------------------------------------------------
# 核心：把所有投影片片段，依序組合成一部完整影片（含真正的交叉轉場）
# ------------------------------------------------------------
def build_slides_with_transitions(
    slide_images: list,
    audio_results: list,
    output_path: str,
    transition: str = "fade",
    transition_duration: float = DEFAULT_TRANSITION_DURATION,
    resolution: tuple = DEFAULT_RESOLUTION,
    fps: int = DEFAULT_FPS,
) -> str:
    """
    把「每頁投影片圖片」+「每頁配音」依序合成一部影片，並在頁與頁之間套用
    真正的 FFmpeg 交叉轉場效果（xfade），而不是每頁各自獨立做淡入淡出特效。

    設計核心（確保「配音播完 → 播放轉場 → 下一張開始」，配音絕對不會被轉場切斷）：
        - 每一頁的配音一定完整播放，不會被轉場效果截斷
        - 轉場動畫發生在「配音播完之後」額外增加的時間，這段時間是真正
          新增到總片長的（每轉場一次，總片長增加 transition_duration 秒），
          不是從配音時間裡面借用
        - 實作方式：每頁片段除了配音本身的畫面時間，會在「配音結束後」
          多留一段 transition_duration 的靜音緩衝畫面（用同一張投影片圖片），
          專門提供給 xfade 轉場效果使用；配音音軌則用 adelay 往後推移到
          正確的位置，確保聲音只出現在真正屬於該頁的時間範圍內
        - 第一頁不需要「進場」緩衝，最後一頁不需要「出場」緩衝

    參數:
        slide_images: 每頁投影片圖片路徑清單
        audio_results: 每頁配音資訊清單 (需含 "audio_path" / "duration")
        output_path: 輸出的 mp4 路徑
        transition: 轉場風格，見 TRANSITION_CHOICES ("none" 代表不使用轉場)
        transition_duration: 轉場時間長度（秒），使用者可自訂
        resolution / fps: 輸出畫質設定

    回傳:
        output_path
    """
    n = min(len(slide_images), len(audio_results))
    if n == 0:
        raise ValueError("沒有可用的投影片圖片或音檔，無法合成影片。")

    width, height = resolution
    fit = _fit_filter(width, height)
    xfade_name = XFADE_TRANSITION_MAP.get(transition)

    durations = [(audio_results[i].get("duration", 0.0) or 1.0) for i in range(n)]

    use_transition = bool(xfade_name) and transition_duration > 0 and n > 1
    d = float(transition_duration) if use_transition else 0.0

    # 每頁片段的「進場緩衝」「出場緩衝」：第一頁沒有進場緩衝，最後一頁沒有出場緩衝
    lead = [d if (use_transition and i > 0) else 0.0 for i in range(n)]
    trail = [d if (use_transition and i < n - 1) else 0.0 for i in range(n)]
    clip_lengths = [durations[i] + lead[i] + trail[i] for i in range(n)]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    cmd = ["ffmpeg", "-y"]
    for i in range(n):
        cmd += ["-loop", "1", "-t", f"{clip_lengths[i]:.3f}", "-i", slide_images[i]]
    for i in range(n):
        cmd += ["-i", audio_results[i]["audio_path"]]

    filter_parts = []
    for i in range(n):
        # 影像輸入是第 i 個(索引0..n-1)
        filter_parts.append(f"[{i}:v]{fit},fps={fps},format=yuv420p,setsar=1[v{i}]")

    for i in range(n):
        # 音訊處理：音訊軌道用「單純依序串接、中間補上靜音間隔」的方式建構，
        # 不需要像影像那樣做前後雙重緩衝 —— 因為 xfade 對影像做的是「重疊」混合
        # (畫面總長 = 個別片段總長 - 重疊部分)，但音訊是直接串接、沒有重疊，
        # 如果audio也比照影像做前後雙重緩衝，會導致音訊總長比畫面總長多算一次
        # 轉場時間（這是實測時抓到的真實 bug，兩者長度對不起來會讓輸出影片
        # 的總長跟著音軌跑，不是預期的影像長度）。
        # 做法：只在「非最後一頁」的音訊尾端補上 D 秒靜音，讓音訊軌道之間
        # 剛好留出跟畫面轉場「同樣長度」的間隔，串接後總長會等於畫面總長。
        pad_target = durations[i] + (d if i < n - 1 else 0.0)
        filter_parts.append(
            f"[{n + i}:a]apad=whole_dur={pad_target:.3f}[a{i}]"
        )

    # --- 影像：串接（若無轉場）或用 xfade 逐一交叉轉場 ---
    if use_transition:
        prev_label = "v0"
        cumulative = clip_lengths[0]
        video_out_label = "v0"
        for i in range(1, n):
            offset = max(cumulative - d, 0.0)
            out_label = f"vt{i}"
            filter_parts.append(
                f"[{prev_label}][v{i}]xfade=transition={xfade_name}:"
                f"duration={d:.3f}:offset={offset:.3f}[{out_label}]"
            )
            prev_label = out_label
            video_out_label = out_label
            cumulative += clip_lengths[i] - d
    else:
        concat_inputs = "".join(f"[v{i}]" for i in range(n))
        filter_parts.append(f"{concat_inputs}concat=n={n}:v=1:a=0[vout]")
        video_out_label = "vout"

    # --- 音訊：單純依序串接即可（沉默間隔已經透過 adelay/apad 處理好了，不需要交叉淡化） ---
    audio_concat_inputs = "".join(f"[a{i}]" for i in range(n))
    filter_parts.append(f"{audio_concat_inputs}concat=n={n}:v=0:a=1[aout]")

    filter_complex = ";".join(filter_parts)

    cmd += [
        "-filter_complex", filter_complex,
        "-map", f"[{video_out_label}]",
        "-map", "[aout]",
        "-c:v", "libx264", "-tune", "stillimage",
        "-c:a", "aac", "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        "-r", str(fps),
        output_path,
    ]

    result = run_subprocess(cmd, description="合成投影片影片（含轉場）")
    if result.returncode != 0:
        raise RuntimeError("影片片段合成/轉場失敗")

    return output_path


# ------------------------------------------------------------
# 字幕燒錄
# ------------------------------------------------------------
def burn_subtitles(
    video_path: str,
    srt_path: str,
    output_path: str,
    font_size: int = 22,
    resolution: tuple = DEFAULT_RESOLUTION,
) -> str:
    """
    把 SRT 字幕「燒錄」進影片畫面中（硬字幕）。
    優點是任何播放器、任何上傳平台（包含企業 LMS）都保證看得到字幕，
    缺點是無法關閉；如果使用者不想要字幕，直接跳過這個步驟即可。

    參數:
        video_path: 尚未加字幕的影片路徑
        srt_path: SRT 字幕檔路徑
        output_path: 輸出（已加字幕）的影片路徑
        font_size: 字幕字級大小（可由使用者在介面上調整）
        resolution: 影片實際解析度 (width, height)。
            重要：FFmpeg 的 subtitles 濾鏡在讀取純文字 SRT（沒有內建 PlayResX/PlayResY
            解析度資訊）時，預設會套用字幕格式舊時代的基準解析度去換算字級，
            跟實際輸出的 1080p 影片對不起來，導致字幕被放大好幾倍、看起來過大。
            這裡明確傳入 original_size=實際影片解析度，讓字級換算基準正確對齊，
            font_size 才會是「所見即所得」的大小。

    回傳:
        output_path
    """
    if not os.path.exists(srt_path):
        raise FileNotFoundError(f"找不到字幕檔：{srt_path}")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # subtitles 濾鏡的檔案路徑在某些系統上對特殊字元(冒號等)敏感，這裡統一轉成絕對路徑，
    # 並用 FFmpeg 濾鏡語法要求的方式跳脫路徑中的反斜線與冒號（Windows 路徑常見）。
    abs_srt_path = os.path.abspath(srt_path).replace("\\", "/").replace(":", "\\:")

    width, height = resolution

    force_style = (
        f"FontName={SUBTITLE_FONT_NAME},FontSize={font_size},"
        "PrimaryColour=&HFFFFFF&,OutlineColour=&H000000&,BorderStyle=1,Outline=2,MarginV=28"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", f"subtitles={abs_srt_path}:original_size={width}x{height}:force_style='{force_style}'",
        "-c:v", "libx264",
        "-c:a", "copy",
        output_path,
    ]
    result = run_subprocess(cmd, description="燒錄字幕進影片")
    if result.returncode != 0:
        raise RuntimeError("字幕燒錄失敗")

    return output_path


# ------------------------------------------------------------
# Step 6：Logo 浮水印
# ------------------------------------------------------------
def add_logo_overlay(
    video_path: str,
    logo_path: str,
    output_path: str,
    logo_width: int = 160,
    margin: int = 24,
) -> str:
    """
    把使用者上傳的 Logo 圖片，以浮水印形式疊加在影片「右上角」。

    參數:
        video_path: 原始影片路徑
        logo_path: Logo 圖片路徑（png/jpg，若為 png 且有透明背景，透明部分會正確顯示）
        output_path: 輸出（已加上 Logo）的影片路徑
        logo_width: Logo 顯示寬度（像素），高度會依原始比例自動縮放
        margin: Logo 距離影片右邊界、上邊界的邊距（像素）

    回傳:
        output_path
    """
    if not os.path.exists(logo_path):
        raise FileNotFoundError(f"找不到 Logo 圖片：{logo_path}")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    filter_complex = (
        f"[1:v]scale={logo_width}:-1[logo];"
        f"[0:v][logo]overlay=W-w-{margin}:{margin}"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-i", logo_path,
        "-filter_complex", filter_complex,
        "-c:a", "copy",
        "-pix_fmt", "yuv420p",
        output_path,
    ]
    result = run_subprocess(cmd, description="加入 Logo 浮水印")
    if result.returncode != 0:
        raise RuntimeError("Logo 浮水印加入失敗")

    return output_path


# ------------------------------------------------------------
# Step 6：背景音樂混音
# ------------------------------------------------------------
def mix_background_music(
    video_path: str,
    bgm_path: str,
    output_path: str,
    bgm_volume: float = 0.2,
) -> str:
    """
    把使用者上傳的背景音樂，與原本的旁白配音混音在一起。
    背景音樂會自動循環播放以填滿整部影片長度，並在影片結束時一併裁切（不會播不完或留空白）。

    參數:
        video_path: 原始影片路徑（已含旁白配音）
        bgm_path: 背景音樂檔案路徑（mp3）
        output_path: 輸出（已混入背景音樂）的影片路徑
        bgm_volume: 背景音樂音量（0.0~1.0），預設 0.2（避免蓋過旁白）

    回傳:
        output_path
    """
    if not os.path.exists(bgm_path):
        raise FileNotFoundError(f"找不到背景音樂檔案：{bgm_path}")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    filter_complex = (
        f"[1:a]volume={bgm_volume}[bgm];"
        f"[0:a][bgm]amix=inputs=2:duration=first:dropout_transition=2[aout]"
    )

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-stream_loop", "-1", "-i", bgm_path,
        "-filter_complex", filter_complex,
        "-map", "0:v", "-map", "[aout]",
        "-c:v", "copy",
        "-c:a", "aac",
        "-shortest",
        output_path,
    ]
    result = run_subprocess(cmd, description="混入背景音樂")
    if result.returncode != 0:
        raise RuntimeError("背景音樂混音失敗")

    return output_path


# ------------------------------------------------------------
# Step 6：片頭 / 片尾串接
# ------------------------------------------------------------
def concatenate_with_intro_outro(
    main_video_path: str,
    output_path: str,
    intro_path: str = None,
    outro_path: str = None,
    resolution: tuple = DEFAULT_RESOLUTION,
    fps: int = DEFAULT_FPS,
) -> str:
    """
    把使用者上傳的「片頭」「片尾」影片，接在主要課程內容的前後。

    因為片頭/片尾是使用者自行上傳的任意規格影片（解析度、幀率、編碼都可能不同），
    這裡使用 FFmpeg 的 concat filter（而非 concat demuxer）：
    concat filter 會先把每段影片統一縮放/補黑邊到相同解析度與幀率，再重新編碼串接，
    確保不同來源的影片也能正確、順暢地接在一起，不會有跳動或黑畫面。

    參數:
        main_video_path: 主要課程內容影片路徑
        output_path: 最終輸出路徑
        intro_path: 選填，片頭影片路徑
        outro_path: 選填，片尾影片路徑
        resolution / fps: 統一輸出的畫質設定

    回傳:
        output_path
    """
    parts = []
    if intro_path and os.path.exists(intro_path):
        parts.append(intro_path)
    parts.append(main_video_path)
    if outro_path and os.path.exists(outro_path):
        parts.append(outro_path)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if len(parts) == 1:
        # 沒有片頭也沒有片尾，直接複製主要內容作為最終輸出
        import shutil
        shutil.copy(main_video_path, output_path)
        return output_path

    width, height = resolution
    inputs = []
    filter_segments = []
    for idx, p in enumerate(parts):
        inputs += ["-i", p]
        filter_segments.append(
            f"[{idx}:v]scale={width}:{height}:force_original_aspect_ratio=decrease,"
            f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color=black,"
            f"fps={fps},setsar=1,format=yuv420p[v{idx}];"
            f"[{idx}:a]aformat=sample_rates=44100:channel_layouts=stereo[a{idx}]"
        )

    concat_inputs = "".join(f"[v{i}][a{i}]" for i in range(len(parts)))
    filter_complex = ";".join(filter_segments) + f";{concat_inputs}concat=n={len(parts)}:v=1:a=1[outv][outa]"

    cmd = ["ffmpeg", "-y"] + inputs + [
        "-filter_complex", filter_complex,
        "-map", "[outv]", "-map", "[outa]",
        "-c:v", "libx264", "-c:a", "aac", "-pix_fmt", "yuv420p",
        output_path,
    ]
    result = run_subprocess(cmd, description="串接片頭/片尾")
    if result.returncode != 0:
        raise RuntimeError("片頭/片尾串接失敗")

    return output_path


# ------------------------------------------------------------
# 整合：完整課程影片合成流程
# ------------------------------------------------------------
def build_full_course_video(
    slide_images: list,
    audio_results: list,
    output_path: str,
    srt_path: str = None,
    subtitle_font_size: int = 22,
    transition: str = "fade",
    transition_duration: float = DEFAULT_TRANSITION_DURATION,
    resolution: tuple = DEFAULT_RESOLUTION,
    fps: int = DEFAULT_FPS,
    temp_dir: str = None,
    logo_path: str = None,
    logo_width: int = 160,
    logo_margin: int = 24,
    bgm_path: str = None,
    bgm_volume: float = 0.2,
    intro_path: str = None,
    outro_path: str = None,
    progress_callback=None,
) -> str:
    """
    完整流程：把「每頁投影片圖片」+「每頁配音」合成完整課程影片，
    並可選擇性加入字幕、Logo浮水印、背景音樂、片頭/片尾。
    這是提供給 app.py 呼叫的主要入口函式。

    參數:
        slide_images: 每頁投影片圖片路徑清單（ppt_reader.load_presentation 的結果）
        audio_results: 每頁配音資訊清單（tts.generate_audio_for_all_slides 的結果，
                       需含 audio_path / duration）
        output_path: 最終輸出的 mp4 路徑
        srt_path: 選填，字幕檔路徑；有提供的話會燒錄進最終影片
                  （注意：這份字幕檔必須是用「同一個 transition_duration」產生的，
                  app.py 會確保這點，見 subtitle.generate_full_srt 的說明）
        subtitle_font_size: 字幕燒錄的字級大小（可依需求調整大小）
        transition: 轉場效果，見 TRANSITION_CHOICES
        transition_duration: 轉場時間長度（秒），使用者可自訂；每頁配音播完後，
                             會多留這麼多秒的轉場緩衝時間才切到下一頁，畫面總長
                             也會因此增加（每次轉場 +transition_duration 秒）
        resolution / fps: 輸出畫質設定
        temp_dir: 存放中間產生的片段檔案的暫存資料夾；未提供則用 output_path 旁邊的暫存資料夾
        logo_path: 選填，Logo 圖片路徑，會疊加在影片右上角
        logo_width / logo_margin: Logo 顯示寬度與邊距
        bgm_path: 選填，背景音樂檔案路徑，會與旁白配音混音
        bgm_volume: 背景音樂音量 (0.0~1.0)
        intro_path: 選填，片頭影片路徑，會接在最前面
        outro_path: 選填，片尾影片路徑，會接在最後面
        progress_callback: 選填，function(fraction: float)，回報進度給 Gradio 進度條

    回傳:
        最終影片的檔案路徑
    """
    if len(slide_images) != len(audio_results):
        log(
            f"⚠️ 投影片圖片數量({len(slide_images)}) 與配音數量({len(audio_results)}) 不一致，"
            "請確認是否已完成 Step1/Step3。",
            "warning",
        )

    if temp_dir is None:
        temp_dir = output_path + "_temp_clips"
    os.makedirs(temp_dir, exist_ok=True)

    total = min(len(slide_images), len(audio_results))

    # --- 階段 1：合成所有投影片頁面（含轉場效果），佔整體進度 0% ~ 60% ---
    log(f"合成 {total} 頁投影片影片中（轉場效果：{transition}，轉場時間：{transition_duration}秒）...")
    current_path = os.path.join(temp_dir, "01_slides_with_transitions.mp4")
    build_slides_with_transitions(
        slide_images[:total], audio_results[:total], current_path,
        transition=transition, transition_duration=transition_duration,
        resolution=resolution, fps=fps,
    )
    if progress_callback:
        progress_callback(0.60)

    # --- 階段 3：字幕燒錄（65% ~ 75%，選填） ---
    if srt_path and os.path.exists(srt_path):
        log("燒錄字幕進影片中...")
        next_path = os.path.join(temp_dir, "02_with_subs.mp4")
        burn_subtitles(current_path, srt_path, next_path, font_size=subtitle_font_size, resolution=resolution)
        current_path = next_path
    if progress_callback:
        progress_callback(0.75)

    # --- 階段 4：Logo 浮水印（75% ~ 82%，選填） ---
    if logo_path and os.path.exists(logo_path):
        log("加入 Logo 浮水印中...")
        next_path = os.path.join(temp_dir, "03_with_logo.mp4")
        add_logo_overlay(current_path, logo_path, next_path, logo_width=logo_width, margin=logo_margin)
        current_path = next_path
    if progress_callback:
        progress_callback(0.82)

    # --- 階段 5：背景音樂混音（82% ~ 90%，選填） ---
    if bgm_path and os.path.exists(bgm_path):
        log("混入背景音樂中...")
        next_path = os.path.join(temp_dir, "04_with_bgm.mp4")
        mix_background_music(current_path, bgm_path, next_path, bgm_volume=bgm_volume)
        current_path = next_path
    if progress_callback:
        progress_callback(0.90)

    # --- 階段 6：片頭 / 片尾（90% ~ 100%，選填） ---
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    if (intro_path and os.path.exists(intro_path)) or (outro_path and os.path.exists(outro_path)):
        log("串接片頭/片尾中...")
        concatenate_with_intro_outro(
            current_path, output_path,
            intro_path=intro_path, outro_path=outro_path,
            resolution=resolution, fps=fps,
        )
    else:
        import shutil
        shutil.copy(current_path, output_path)

    if progress_callback:
        progress_callback(1.0)

    # 清理中間產生的暫存片段檔案，避免佔用空間
    import shutil
    shutil.rmtree(temp_dir, ignore_errors=True)

    log(f"✅ 課程影片合成完成：{output_path}")
    return output_path


# ------------------------------------------------------------
# 自我測試區塊
# ------------------------------------------------------------
if __name__ == "__main__":
    """
    測試整個影片合成流程：
    1. 建立測試投影片圖片（3頁，用不同顏色/圖案區分）與對應的靜音配音
    2. 測試所有轉場風格（none/fade/crossfade/dissolve/slideleft/slideright/zoom），
       驗證畫面總長 = 配音總長 + (頁數-1)×轉場時間
    3. 測試字幕燒錄
    4. 測試完整流程 build_full_course_video()
    5. 測試 Logo / 背景音樂 / 片頭片尾
    6. 測試完整流程（含全部選項 + 轉場時間）
    每一步都用 ffprobe 驗證輸出的影片時長、解析度是否符合預期。
    """
    import shutil

    test_dir = "/home/claude/PPT2Course_AI/_test_video"
    os.makedirs(test_dir, exist_ok=True)

    def get_duration(path):
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        )
        return float(result.stdout.strip())

    def get_resolution(path):
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=width,height",
             "-of", "csv=s=x:p=0", path],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        )
        w, h = result.stdout.strip().split("x")
        return int(w), int(h)

    # --- 準備測試素材 ---
    print("===== 準備測試素材 =====")
    test_images = []
    for i, color in enumerate(["red", "green", "blue"], start=1):
        img_path = os.path.join(test_dir, f"slide_{i}.png")
        subprocess.run(
            ["ffmpeg", "-y", "-f", "lavfi", "-i", f"testsrc=size=1024x768:rate=1",
             "-frames:v", "1", img_path],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        test_images.append(img_path)

    test_audio_results = []
    for i, dur in enumerate([2.0, 1.5, 2.5], start=1):
        audio_path = os.path.join(test_dir, f"audio_{i}.mp3")
        subprocess.run(
            ["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=r=24000:cl=mono",
             "-t", str(dur), "-q:a", "9", audio_path],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        test_audio_results.append({"slide_index": i, "audio_path": audio_path, "duration": dur, "has_audio": True})

    print(f"準備了 {len(test_images)} 張測試圖片與音檔\n")

    # --- 測試 1：所有轉場風格各自合成一段完整（3頁）影片 ---
    print("===== 測試 1：所有轉場風格 =====")
    D = 0.5
    for transition in TRANSITION_CHOICES:
        out_path = os.path.join(test_dir, f"test_{transition}.mp4")
        build_slides_with_transitions(
            test_images, test_audio_results, out_path,
            transition=transition, transition_duration=D,
            resolution=(1280, 720), fps=30,
        )
        dur = get_duration(out_path)
        res = get_resolution(out_path)
        n = len(test_images)
        Dused = D if transition != "none" else 0.0
        expected_dur = sum(a["duration"] for a in test_audio_results) + Dused * (n - 1)
        print(f"轉場={transition}: 時長={dur:.2f}秒 (預期{expected_dur:.2f}秒), 解析度={res}")
        assert abs(dur - expected_dur) < 0.2, f"轉場 {transition} 的時長不符預期！"
        assert res == (1280, 720), f"轉場 {transition} 的解析度不符預期！"
    print("✅ 所有轉場風格測試通過（畫面時長 = 配音總長 + (頁數-1)×轉場時間）\n")

    concat_out = os.path.join(test_dir, "test_fade.mp4")  # 沿用上面 fade 轉場的輸出，供後續測試使用
    concat_dur = get_duration(concat_out)

    # --- 測試 2：字幕燒錄 ---
    print("===== 測試 2：字幕燒錄 =====")
    test_srt_path = os.path.join(test_dir, "test.srt")
    with open(test_srt_path, "w", encoding="utf-8") as f:
        f.write("1\n00:00:00,000 --> 00:00:02,000\n測試字幕內容\n\n2\n00:00:02,000 --> 00:00:04,000\n第二段字幕\n")

    sub_out = os.path.join(test_dir, "test_with_sub.mp4")
    burn_subtitles(concat_out, test_srt_path, sub_out)
    assert os.path.exists(sub_out)
    print("✅ 字幕燒錄測試通過\n")

    # --- 測試 3：完整流程 build_full_course_video()（含轉場時間） ---
    print("===== 測試 3：完整流程 build_full_course_video() =====")
    final_output = os.path.join(test_dir, "final_course.mp4")
    build_full_course_video(
        test_images, test_audio_results, final_output,
        srt_path=test_srt_path, transition="fade", transition_duration=D,
        resolution=(1280, 720), fps=30,
    )
    final_dur = get_duration(final_output)
    n = len(test_images)
    expected_total = sum(a["duration"] for a in test_audio_results) + D * (n - 1)
    print(f"完整課程影片時長：{final_dur:.2f}秒（預期約 {expected_total:.1f} 秒，含轉場時間）")
    assert os.path.exists(final_output)
    assert abs(final_dur - expected_total) < 0.5, "完整影片總時長與預期（配音總長+轉場時間）不符！"
    print("✅ 完整流程測試通過（已確認轉場時間有正確計入總長）")

    # --- 測試 5：Logo 浮水印 ---
    print("\n===== 測試 5：Logo 浮水印 =====")
    logo_path = os.path.join(test_dir, "logo.png")
    subprocess.run(
        ["ffmpeg", "-y", "-f", "lavfi", "-i", "color=c=red:s=200x200",
         "-frames:v", "1", logo_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    logo_out = os.path.join(test_dir, "test_with_logo.mp4")
    add_logo_overlay(concat_out, logo_path, logo_out, logo_width=160, margin=24)
    assert os.path.exists(logo_out)
    logo_res = get_resolution(logo_out)
    print(f"加Logo後解析度：{logo_res}（應與原影片相同）")
    assert logo_res == get_resolution(concat_out), "加Logo後解析度不應改變！"
    print("✅ Logo 浮水印測試通過")

    # --- 測試 6：背景音樂混音 ---
    print("\n===== 測試 6：背景音樂混音 =====")
    bgm_path = os.path.join(test_dir, "bgm.mp3")
    subprocess.run(
        ["ffmpeg", "-y", "-f", "lavfi", "-i", "sine=frequency=440:duration=2",
         "-q:a", "9", bgm_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    bgm_out = os.path.join(test_dir, "test_with_bgm.mp4")
    mix_background_music(concat_out, bgm_path, bgm_out, bgm_volume=0.3)
    assert os.path.exists(bgm_out)
    bgm_dur = get_duration(bgm_out)
    print(f"混入背景音樂後時長：{bgm_dur:.2f}秒（應與原影片時長一致，約{concat_dur:.2f}秒）")
    assert abs(bgm_dur - concat_dur) < 0.3, "混音後影片時長不應改變（背景音樂應循環並裁切至相同長度）！"
    print("✅ 背景音樂混音測試通過")

    # --- 測試 7：片頭/片尾串接（模擬不同解析度的來源，驗證統一畫質功能） ---
    print("\n===== 測試 7：片頭/片尾串接 =====")
    intro_path = os.path.join(test_dir, "intro.mp4")
    outro_path = os.path.join(test_dir, "outro.mp4")
    # 故意用不同解析度/幀率產生片頭片尾，驗證 concat filter 能正確統一規格
    subprocess.run(
        ["ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc=size=640x480:rate=25:duration=1",
         "-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo", "-shortest", intro_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    subprocess.run(
        ["ffmpeg", "-y", "-f", "lavfi", "-i", "testsrc=size=800x600:rate=15:duration=1",
         "-f", "lavfi", "-i", "anullsrc=r=22050:cl=mono", "-shortest", outro_path],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    intro_outro_out = os.path.join(test_dir, "test_with_intro_outro.mp4")
    concatenate_with_intro_outro(
        concat_out, intro_outro_out,
        intro_path=intro_path, outro_path=outro_path,
        resolution=(1280, 720), fps=30,
    )
    assert os.path.exists(intro_outro_out)
    io_dur = get_duration(intro_outro_out)
    io_res = get_resolution(intro_outro_out)
    expected_io_dur = 1.0 + concat_dur + 1.0  # intro(1s) + 主內容 + outro(1s)
    print(f"片頭+主內容+片尾 總時長：{io_dur:.2f}秒（預期約 {expected_io_dur:.2f} 秒），解析度：{io_res}")
    assert abs(io_dur - expected_io_dur) < 0.5, "片頭/片尾串接後總時長不符預期！"
    assert io_res == (1280, 720), "片頭/片尾串接後解析度應統一為指定解析度！"
    print("✅ 片頭/片尾串接測試通過（含不同規格來源自動統一畫質）")

    # --- 測試 8：完整流程（含全部Step6選項 + 轉場時間） ---
    print("\n===== 測試 8：完整流程（含Logo+BGM+片頭片尾+轉場） =====")
    final_full_output = os.path.join(test_dir, "final_full_course.mp4")
    build_full_course_video(
        test_images, test_audio_results, final_full_output,
        srt_path=test_srt_path, transition="zoom", transition_duration=D,
        resolution=(1280, 720), fps=30,
        logo_path=logo_path, bgm_path=bgm_path, bgm_volume=0.25,
        intro_path=intro_path, outro_path=outro_path,
    )
    assert os.path.exists(final_full_output)
    full_dur = get_duration(final_full_output)
    n = len(test_images)
    expected_full_dur = 1.0 + (sum(a["duration"] for a in test_audio_results) + D * (n - 1)) + 1.0
    print(f"完整課程（含Logo/BGM/片頭片尾/轉場）總時長：{full_dur:.2f}秒（預期約 {expected_full_dur:.2f} 秒）")
    assert abs(full_dur - expected_full_dur) < 0.6, "完整流程總時長與預期不符！"
    print("✅ 完整流程（含Step6全部選項+轉場時間）測試通過")

    # 清理測試檔案
    shutil.rmtree(test_dir, ignore_errors=True)
    print("\n🎉 所有影片合成測試通過！")


Writing video.py


In [10]:
%%writefile ui_theme.py
# -*- coding: utf-8 -*-
"""
ui_theme.py
===========
集中管理 PPT2Course Lab 網頁介面的視覺樣式：
    1. Gradio 主題色彩（紫色系，透過官方 gr.themes API 設定，非硬改內部 class）
    2. 自訂 CSS（卡片樣式、陰影、圓角、動畫、RWD、懸浮說明按鈕與 Modal）
    3. 使用手冊（新手教學 + FAQ）的 HTML 內容

app.py 只需要：
    from ui_theme import build_theme, CUSTOM_CSS, HELP_FAB_HTML
    gr.Blocks(theme=build_theme(), css=CUSTOM_CSS)
    gr.HTML(HELP_FAB_HTML)

⚠️ 這個檔案只負責「畫面長什麼樣子」，完全不碰任何後端邏輯、資料處理、
   PPT解析、Gemini API、講稿/配音/字幕/影片產生流程。
"""

import gradio as gr


# ============================================================
# 1. 色彩定義（對應需求文件的色票）
# ============================================================
COLOR_PRIMARY = "#5B3FD6"
COLOR_PRIMARY_HOVER = "#6D52E5"
COLOR_PRIMARY_DARK = "#4328A8"
COLOR_BG = "#FFFFFF"  # 頁面整體背景（依需求全面改為白色，不再使用淡紫色）
COLOR_CARD = "#FFFFFF"
COLOR_BORDER = "#E6E2F5"
COLOR_SUCCESS = "#22C55E"
COLOR_WARNING = "#F59E0B"
COLOR_ERROR = "#EF4444"
COLOR_TEXT_TITLE = "#1F1F2E"
COLOR_TEXT_BODY = "#4B4B63"
COLOR_TEXT_SECONDARY = "#7A7A93"


def build_theme() -> gr.Theme:
    """
    使用 Gradio 官方的 Theme API 建立紫色主題。
    比起直接覆寫 Gradio 內部 CSS class（容易隨版本改變而失效），
    透過官方 gr.themes.Color / .set() API 設定更穩定、也是 Gradio 官方建議的自訂主題方式。
    """
    purple_palette = gr.themes.Color(
        name="ptc_purple",
        c50="#F7F6FC",
        c100="#EFEBFA",
        c200="#DED5F5",
        c300="#C4B8ED",
        c400="#9F86E0",
        c500=COLOR_PRIMARY,
        c600=COLOR_PRIMARY_HOVER,
        c700="#4F37C0",
        c800=COLOR_PRIMARY_DARK,
        c900="#361F82",
        c950="#241456",
    )

    theme = gr.themes.Soft(
        primary_hue=purple_palette,
        neutral_hue=gr.themes.colors.slate,
        radius_size=gr.themes.sizes.radius_md,  # 圓角統一 10~14px 區間，屬建構子(constructor)參數
        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    ).set(
        # --- 按鈕 ---
        button_primary_background_fill=COLOR_PRIMARY,
        button_primary_background_fill_hover=COLOR_PRIMARY_HOVER,
        button_primary_text_color="#FFFFFF",
        button_primary_border_color=COLOR_PRIMARY,
        button_secondary_background_fill="#FFFFFF",
        button_secondary_border_color=COLOR_BORDER,
        button_secondary_text_color=COLOR_TEXT_TITLE,
        # --- 版面底色 / 區塊 ---
        # 注意：background_fill_secondary 原本設為淡紫色會讓卡片內的 Markdown/提示文字
        # 出現「反白」的淡紫色區塊，跟白色卡片不一致、看起來髒髒的。
        # 這裡改成白色，讓卡片「內部」統一是乾淨的白底，只有最外層頁面背景維持淡紫色。
        body_background_fill=COLOR_BG,
        background_fill_primary=COLOR_CARD,
        background_fill_secondary=COLOR_CARD,
        border_color_primary=COLOR_BORDER,
        block_background_fill=COLOR_CARD,
        block_border_color=COLOR_BORDER,
        block_shadow="none",
        # --- 文字 ---
        body_text_color=COLOR_TEXT_BODY,
        block_title_text_color=COLOR_TEXT_TITLE,
        block_label_text_color=COLOR_TEXT_SECONDARY,
        block_label_background_fill="#FFFFFF",
        panel_background_fill="#FFFFFF",
        # --- 表單元件（Slider / Radio / Checkbox / Dropdown 等） ---
        input_background_fill="#FFFFFF",
        input_border_color=COLOR_BORDER,
        checkbox_background_color_selected=COLOR_PRIMARY,
        checkbox_border_color_selected=COLOR_PRIMARY,
        slider_color=COLOR_PRIMARY,
        # Radio / Checkbox「選項標籤」的樣式（例如旁白模式的四個選項）：
        # 統一白底 + 深色文字，選中時邊框變紫色、文字變紫色，而不是整個底色反白，
        # 這樣才不會發生「選中時底色跟文字顏色一樣，看不到字」的問題。
        checkbox_label_background_fill="#FFFFFF",
        checkbox_label_background_fill_hover="#FFFFFF",
        checkbox_label_background_fill_selected="#FFFFFF",
        checkbox_label_border_color=COLOR_BORDER,
        checkbox_label_border_color_hover=COLOR_PRIMARY,
        checkbox_label_border_color_selected=COLOR_PRIMARY,
        checkbox_label_text_color=COLOR_TEXT_BODY,
        checkbox_label_text_color_selected=COLOR_PRIMARY,
        # --- 進度條（部分 Gradio 版本會用 loader/progress 相關 token） ---
        loader_color=COLOR_PRIMARY,
    )
    return theme


# ============================================================
# 2. 自訂 CSS：卡片樣式 / 動畫 / RWD / 懸浮說明按鈕與 Modal
# ============================================================
CUSTOM_CSS = f"""
:root {{
    --ptc-primary: {COLOR_PRIMARY};
    --ptc-primary-hover: {COLOR_PRIMARY_HOVER};
    --ptc-primary-dark: {COLOR_PRIMARY_DARK};
    --ptc-bg: {COLOR_BG};
    --ptc-card: {COLOR_CARD};
    --ptc-border: {COLOR_BORDER};
    --ptc-success: {COLOR_SUCCESS};
    --ptc-warning: {COLOR_WARNING};
    --ptc-error: {COLOR_ERROR};
    --ptc-title: {COLOR_TEXT_TITLE};
    --ptc-body: {COLOR_TEXT_BODY};
    --ptc-secondary: {COLOR_TEXT_SECONDARY};
}}

/* ------------------------------------------------------------ */
/* 整體背景 / 字體                                                */
/* ------------------------------------------------------------ */
.gradio-container {{
    background: var(--ptc-bg) !important;
}}

/* ------------------------------------------------------------ */
/* 卡片樣式：套用在每個 Step 的 gr.Group（elem_classes=["ptc-card"]） */
/* ------------------------------------------------------------ */
.ptc-card {{
    background: var(--ptc-card) !important;
    border: 1px solid var(--ptc-border) !important;
    border-radius: 14px !important;
    box-shadow: 0 1px 3px rgba(31, 31, 46, 0.05), 0 6px 18px rgba(91, 63, 214, 0.04) !important;
    padding: 18px 22px !important;
    margin-bottom: 14px !important;
    transition: box-shadow 0.2s ease, transform 0.15s ease;
}}
.ptc-card:hover {{
    box-shadow: 0 4px 14px rgba(91, 63, 214, 0.10), 0 10px 28px rgba(91, 63, 214, 0.06) !important;
}}

/* Step 標題文字 */
.ptc-step-title {{
    font-weight: 600;
    font-size: 1.05rem;
    color: var(--ptc-title);
    margin-bottom: 4px;
}}
.ptc-step-sub {{
    color: var(--ptc-secondary);
    font-size: 0.85rem;
    margin-bottom: 10px;
}}

/* 卡片內部的文字/說明區塊一律透明背景、無陰影、無邊框，
   避免 Markdown/HTML 元件各自帶著淡紫色反白或白底陰影，
   造成「卡片裡面又有小卡片」的雜亂感，讓卡片內部視覺統一、乾淨。
   （其餘區塊的白底，統一交給下方 build_theme() 裡的官方主題變數處理，
   不再用大範圍的 CSS 選擇器強制套用，避免波及 Radio/Checkbox 選中狀態的顏色對比。） */
.ptc-card .prose,
.ptc-card .markdown,
.ptc-card [class*="markdown"] {{
    background: transparent !important;
    box-shadow: none !important;
    border: none !important;
}}

/* 保險規則：不管 Gradio 版本內部怎麼命名 CSS 變數，
   只要是「被選中的 Radio / Checkbox 選項標籤」，一律強制白底 + 紫色文字，
   確保絕對不會發生「選中狀態底色跟文字同色、看不到字」的狀況。 */
.ptc-card label:has(input:checked) {{
    background-color: #FFFFFF !important;
    color: var(--ptc-primary) !important;
    border-color: var(--ptc-primary) !important;
}}
.ptc-card label:has(input:checked) * {{
    color: var(--ptc-primary) !important;
}}

/* 輕量提示文字（例如 API Key 申請連結、參考資料說明）：
   縮小字級、降低顏色對比，讓它是「補充說明」而不是搶眼的區塊，
   畫面才不會看起來塞滿滿。 */
.ptc-hint {{
    color: var(--ptc-secondary) !important;
    font-size: 0.8rem !important;
    line-height: 1.5 !important;
    margin: 2px 0 10px 0 !important;
}}
.ptc-hint a {{
    color: var(--ptc-primary) !important;
}}

/* ------------------------------------------------------------ */
/* 下載按鈕：改用 gr.DownloadButton（本身就是一顆按鈕，沒有檔案圖示、           */
/* 沒有檔名/檔案大小這些多餘資訊列，天生乾淨），這裡只做外觀微調                 */
/* ------------------------------------------------------------ */
.ptc-download {{
    width: 100% !important;
}}
.ptc-download button {{
    min-height: 46px !important;
    font-size: 1rem !important;
    font-weight: 600 !important;
    border-radius: 12px !important;
}}

/* ------------------------------------------------------------ */
/* 上傳區塊：統一深紫色主題，第一次使用的人一眼就能看出「這裡要上傳檔案」        */
/* ------------------------------------------------------------ */
.ptc-upload {{
    border: 2px dashed var(--ptc-primary) !important;
    border-radius: 12px !important;
    background: rgba(91, 63, 214, 0.05) !important;
    transition: background 0.18s ease, border-color 0.18s ease, box-shadow 0.18s ease;
}}
.ptc-upload:hover {{
    background: rgba(91, 63, 214, 0.1) !important;
    border-color: var(--ptc-primary-dark) !important;
    box-shadow: 0 4px 16px rgba(91, 63, 214, 0.18) !important;
}}
/* 拖曳檔案進入時的高亮效果（Gradio 會在拖曳中的元件加上 drag 相關 class） */
.ptc-upload.drag,
.ptc-upload[class*="drag"] {{
    background: rgba(91, 63, 214, 0.16) !important;
    border-color: var(--ptc-primary) !important;
    box-shadow: 0 0 0 3px rgba(91, 63, 214, 0.15) !important;
}}
.ptc-upload [class*="wrap"] {{
    font-size: 1rem !important;
    color: var(--ptc-primary-dark) !important;
}}
/* 中央的上傳箭頭 icon 保留、放大；但「標籤列」左側那個小文件圖示不需要，予以隱藏 */
.ptc-upload svg {{
    width: 34px !important;
    height: 34px !important;
    color: var(--ptc-primary) !important;
}}
.ptc-upload label svg,
.ptc-upload [data-testid="block-label"] svg,
.ptc-upload [class*="label"] svg {{
    display: none !important;
}}

/* 每個上傳區塊上方的小標題（例如「📑 PPT 檔案」），改成更醒目的徽章樣式：
   左側色條 + 淡紫底色，一眼就能跟旁邊的說明文字或提示文字區分開來 */
.ptc-upload-title {{
    font-weight: 700;
    font-size: 1rem;
    color: var(--ptc-primary-dark);
    margin: 4px 0 8px 0;
    padding: 6px 12px;
    border-left: 4px solid var(--ptc-primary);
    background: rgba(91, 63, 214, 0.08);
    border-radius: 0 8px 8px 0;
    display: inline-block;
}}

/* 上傳成功後的提示文字（✅ 已成功上傳：檔名） */
.ptc-upload-notice {{
    color: var(--ptc-success) !important;
    font-size: 0.85rem !important;
    font-weight: 600 !important;
    margin: 2px 0 8px 0 !important;
}}

/* ------------------------------------------------------------ */
/* 尚未完成前置步驟時，按鈕呈現明顯的「未啟用」灰階樣式，
   引導使用者依序操作，不會不知道下一步該按哪裡。 */
/* ------------------------------------------------------------ */
button:disabled,
button[disabled] {{
    background: #E9E9F0 !important;
    color: #ABABC0 !important;
    border-color: #E9E9F0 !important;
    cursor: not-allowed !important;
    box-shadow: none !important;
    transform: none !important;
    filter: none !important;
}}

/* ------------------------------------------------------------ */
/* 按鈕動畫                                                       */
/* ------------------------------------------------------------ */
button {{
    transition: transform 0.12s ease, filter 0.12s ease, box-shadow 0.12s ease !important;
    border-radius: 10px !important;
}}
button:hover {{
    transform: translateY(-1px);
    filter: brightness(1.03);
}}
button:active {{
    transform: translateY(0px);
}}
.ptc-card button.primary {{
    padding: 12px 22px !important;
    font-size: 1rem !important;
    font-weight: 600 !important;
}}

/* ------------------------------------------------------------ */
/* Progress bar 顏色（新版 Gradio 進度條/載入動畫） */
/* ------------------------------------------------------------ */
.progress-bar, .meta-text, [class*="progress"] {{
    color: var(--ptc-primary) !important;
}}

/* ------------------------------------------------------------ */
/* 懸浮說明按鈕 (FAB) + Modal（純 CSS，不依賴 JS，任何 Gradio 版本都能運作） */
/* ------------------------------------------------------------ */
.ptc-help-toggle-input {{
    display: none;
}}

.ptc-fab {{
    position: fixed;
    right: 24px;
    bottom: 24px;
    width: 52px;
    height: 52px;
    border-radius: 50%;
    background: #FFFFFF;
    border: 2px solid var(--ptc-primary);
    color: var(--ptc-primary);
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 22px;
    font-weight: 700;
    cursor: pointer;
    box-shadow: 0 4px 14px rgba(91, 63, 214, 0.28);
    z-index: 10000;
    transition: transform 0.15s ease, background 0.15s ease, color 0.15s ease;
    user-select: none;
}}
.ptc-fab:hover {{
    background: var(--ptc-primary);
    color: #FFFFFF;
    transform: scale(1.06);
}}

.ptc-modal-overlay {{
    display: none;
    position: fixed;
    inset: 0;
    z-index: 10001;
}}
.ptc-help-toggle-input:checked ~ .ptc-modal-overlay {{
    display: block;
    animation: ptc-fade-in 0.18s ease;
}}
.ptc-modal-overlay-bg {{
    position: absolute;
    inset: 0;
    background: rgba(31, 31, 46, 0.5);
    cursor: pointer;
}}
.ptc-modal {{
    position: absolute;
    top: 50%;
    left: 50%;
    transform: translate(-50%, -50%);
    width: min(640px, 90vw);
    max-height: 82vh;
    overflow-y: auto;
    background: #FFFFFF;
    border-radius: 16px;
    padding: 28px 30px 32px;
    box-shadow: 0 24px 70px rgba(31, 31, 46, 0.32);
    font-family: 'Inter', ui-sans-serif, system-ui, sans-serif;
    color: var(--ptc-body);
}}
.ptc-modal-close {{
    position: absolute;
    top: 16px;
    right: 18px;
    width: 32px;
    height: 32px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    background: var(--ptc-bg);
    color: var(--ptc-secondary);
    cursor: pointer;
    font-size: 16px;
    transition: background 0.15s ease, color 0.15s ease;
}}
.ptc-modal-close:hover {{
    background: var(--ptc-primary);
    color: #FFFFFF;
}}

.ptc-modal h2 {{
    color: var(--ptc-title);
    font-size: 1.3rem;
    margin: 0 0 14px 0;
}}
.ptc-modal h3 {{
    color: var(--ptc-primary-dark);
    font-size: 1.02rem;
    margin: 22px 0 10px 0;
}}
.ptc-modal p, .ptc-modal li {{
    font-size: 0.92rem;
    line-height: 1.65;
    color: var(--ptc-body);
}}
.ptc-step-list {{
    list-style: none;
    padding: 0;
    margin: 0;
}}
.ptc-step-list li {{
    display: flex;
    gap: 10px;
    align-items: flex-start;
    padding: 8px 0;
    border-bottom: 1px dashed var(--ptc-border);
}}
.ptc-step-list li:last-child {{
    border-bottom: none;
}}
.ptc-step-num {{
    flex: 0 0 auto;
    width: 24px;
    height: 24px;
    border-radius: 50%;
    background: var(--ptc-primary);
    color: #fff;
    font-size: 0.78rem;
    font-weight: 700;
    display: flex;
    align-items: center;
    justify-content: center;
}}
.ptc-faq-item {{
    margin-bottom: 14px;
}}
.ptc-faq-q {{
    font-weight: 600;
    color: var(--ptc-title);
    margin-bottom: 3px;
}}
.ptc-faq-a {{
    color: var(--ptc-body);
    font-size: 0.9rem;
    line-height: 1.6;
}}

@keyframes ptc-fade-in {{
    from {{ opacity: 0; }}
    to {{ opacity: 1; }}
}}

/* ------------------------------------------------------------ */
/* RWD：平板 / 手機縮小邊距與字級                                   */
/* ------------------------------------------------------------ */
@media (max-width: 1024px) {{
    .ptc-card {{
        padding: 16px 16px !important;
    }}
}}
@media (max-width: 640px) {{
    .ptc-modal {{
        padding: 22px 18px 26px;
        width: 94vw;
    }}
    .ptc-fab {{
        right: 16px;
        bottom: 16px;
        width: 46px;
        height: 46px;
        font-size: 19px;
    }}
}}

/* ------------------------------------------------------------ */
/* 頂部步驟導覽條：讓使用者一眼看懂整體流程有幾步、目前大概在哪 */
/* ------------------------------------------------------------ */
.ptc-steps-bar {{
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    margin: 4px 0 18px 0;
}}
.ptc-steps-bar .ptc-step-chip {{
    display: flex;
    align-items: center;
    gap: 6px;
    background: #FFFFFF;
    border: 1px solid var(--ptc-border);
    border-radius: 999px;
    padding: 6px 14px;
    font-size: 0.82rem;
    color: var(--ptc-secondary);
    white-space: nowrap;
}}
.ptc-steps-bar .ptc-step-chip .ptc-step-num {{
    width: 20px;
    height: 20px;
    font-size: 0.72rem;
}}
.ptc-steps-bar .ptc-step-arrow {{
    color: var(--ptc-border);
    align-self: center;
    font-size: 0.9rem;
}}
@media (max-width: 900px) {{
    .ptc-steps-bar {{
        display: none; /* 手機/小螢幕收起，避免佔用太多空間 */
    }}
}}
"""


# ============================================================
# 3. 使用手冊內容（懸浮按鈕點擊後彈出的 Modal HTML）
# ============================================================
_MANUAL_STEPS = [
    ("上傳 PPT", "在「1️⃣ 上傳並解析 PPT」卡片中，上傳您的 .pptx 檔案，點「解析 PPT」。"),
    ("選擇旁白來源", "在「2️⃣ 產生講稿」選擇：AI 自動生成 / PowerPoint 備忘稿 / 自己的 Word 或 TXT 講稿 / AI 修飾自己的講稿。"),
    ("（如使用AI）選擇 AI 模型與參考資料", "使用 AI 生成/修飾講稿時，需輸入 Gemini API Key，也可以選擇上傳參考資料讓內容更真實。"),
    ("按下生成講稿", "點「✍️ 生成講稿」，系統會依您選擇的方式產生每頁講稿。"),
    ("確認講稿", "在講稿預覽框中檢查、直接編輯文字內容，滿意後可另存為 Word 檔。"),
    ("選擇語音", "在「3️⃣ AI 配音」選擇聲音、調整語速與音量。"),
    ("開始生成配音", "點「🔊 產生配音」，系統會逐頁合成語音，並顯示進度。"),
    ("下載成果", "依序完成字幕（Step4）、影片合成（Step5）後，即可下載 MP3 音檔、講稿 Word、字幕 SRT，以及完整課程 MP4 影片。"),
]

_FAQ_ITEMS = [
    ("沒有 Speaker Notes 怎麼辦？", "可以直接選擇「② AI 自動生成講稿」，系統會依投影片文字內容自動撰寫口語講稿，不需要 PPT 裡有備忘稿。"),
    ("Word 講稿可以放什麼格式？", "支援 .docx 檔案，建議在每一頁講稿前加上「第1頁」「第2頁」等標記，系統會自動依標記對應到正確的投影片頁數。"),
    ("TXT 要怎麼分頁？", "同樣建議用「第1頁」「第2頁」等文字標記分頁；如果沒有標記，系統會改用「空白行」自動分段對應頁數，但建議加上標記以確保完全對應正確。"),
    ("AI 生成需要多久？", "依投影片頁數而定，通常每頁數秒到十幾秒；若使用免費版 Gemini API Key，因為有每分鐘請求數限制，頁數較多時系統會自動排隊等待重試，整體時間會拉長，屬正常現象。"),
    ("生成失敗怎麼辦？", "請先展開畫面上的「📋 即時執行紀錄 (Log)」查看詳細錯誤訊息，常見原因包含 API Key 未填寫或已過期、免費額度用罄、網路不穩定；多數情況下重新點一次生成按鈕即可。"),
]

_steps_html = "".join(
    f"""
    <li>
        <div class="ptc-step-num">{i}</div>
        <div><strong>{title}</strong><br/>{desc}</div>
    </li>
    """
    for i, (title, desc) in enumerate(_MANUAL_STEPS, start=1)
)

_faq_html = "".join(
    f"""
    <div class="ptc-faq-item">
        <div class="ptc-faq-q">Q：{q}</div>
        <div class="ptc-faq-a">A：{a}</div>
    </div>
    """
    for q, a in _FAQ_ITEMS
)

_STEP_LABELS = ["上傳PPT", "產生講稿", "生成配音", "產生字幕", "合成影片"]
_step_chips = []
for i, label in enumerate(_STEP_LABELS, start=1):
    _step_chips.append(
        f'<div class="ptc-step-chip"><span class="ptc-step-num">{i}</span>{label}</div>'
    )
    if i < len(_STEP_LABELS):
        _step_chips.append('<span class="ptc-step-arrow">›</span>')

STEPS_BAR_HTML = '<div class="ptc-steps-bar">' + "".join(_step_chips) + "</div>"


HELP_FAB_HTML = f"""
<input type="checkbox" id="ptc-help-toggle" class="ptc-help-toggle-input">
<label for="ptc-help-toggle" class="ptc-fab" title="使用手冊">?</label>

<div class="ptc-modal-overlay">
    <label for="ptc-help-toggle" class="ptc-modal-overlay-bg"></label>
    <div class="ptc-modal">
        <label for="ptc-help-toggle" class="ptc-modal-close">&#10005;</label>

        <h2>📘 PPT2Course AI 使用教學</h2>

        <ul class="ptc-step-list">
            {_steps_html}
        </ul>

        <h3>❓ 常見問題 FAQ</h3>
        {_faq_html}
    </div>
</div>
"""


Writing ui_theme.py


In [11]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""
app.py
======
PPT2Course AI 系統的主程式進入點，使用 Gradio 建立網頁介面。

【目前進度：Step 6（完整版）】
Step 1：上傳 PPT → 解析內容 → 預覽（頁數 / 圖片 / 文字 / 備忘稿）
Step 2：選擇四種旁白模式之一 → 產生每頁講稿 → 可直接編輯 → 下載成 Word
Step 3：AI 配音（Edge-TTS，免費）→ 可調整聲音/語速/音量 → 逐頁試聽（含進度條）
Step 4：產生字幕（SRT）→ 依配音實際時長自動對齊時間軸
Step 5：影片合成（FFmpeg）→ 圖片+配音+字幕 → 完整 MP4 課程影片，可選轉場效果（含進度條）
Step 6：Logo 浮水印 / 背景音樂 / 片頭片尾（於 Step5 區塊內的「進階選項」）

介面視覺風格（紫色主題、卡片化、動畫、使用手冊懸浮按鈕）統一定義在 ui_theme.py，
本檔案（app.py）只負責畫面結構組裝與呼叫後端函式，不包含樣式邏輯。
"""

import os
import gradio as gr

from utils import (
    ensure_all_dirs,
    check_environment,
    make_session_dir,
    safe_filename,
    natural_sort_key,
    log,
    get_log_text,
    clear_log,
    DIR_PPT,
    DIR_IMAGES,
    DIR_SCRIPT,
    DIR_OUTPUT,
    DIR_AUDIO,
)
from ppt_reader import load_presentation, read_ppt_slides, convert_ppt_to_images
from script_generator import (
    generate_all_scripts,
    format_scripts_for_display,
    parse_scripts_from_display,
    save_scripts_to_docx,
)
from tts import (
    VOICE_OPTIONS,
    DEFAULT_VOICE_LABEL,
    generate_audio_for_all_slides,
    format_audio_summary,
)
from subtitle import generate_full_srt, save_srt_file
from video import (
    build_full_course_video,
    TRANSITION_CHOICES,
    TRANSITION_DURATION_CHOICES,
    DEFAULT_TRANSITION_DURATION,
)
from ui_theme import build_theme, CUSTOM_CSS, HELP_FAB_HTML, STEPS_BAR_HTML


# ------------------------------------------------------------
# 初始化：確保資料夾存在、檢查環境
# ------------------------------------------------------------
ensure_all_dirs()
ENV_STATUS = check_environment()

# 旁白模式：畫面上顯示的中文選項 <-> 程式內部使用的英文代碼
MODE_LABEL_TO_CODE = {
    "① 使用自己的講稿 (上傳 docx / txt)": "own",
    "② AI 自動生成講稿 (需要 Gemini API Key)": "ai_auto",
    "③ 使用 PowerPoint 備忘稿 (Speaker Notes)": "notes",
    "④ AI 修飾自己的講稿 (上傳講稿 + Gemini API Key)": "ai_polish",
}

# 轉場效果：畫面上顯示的中文選項 <-> video.py 內部使用的代碼
TRANSITION_LABEL_TO_CODE = {
    "無轉場（直接切換）": "none",
    "Fade（淡出淡入）": "fade",
    "Cross Fade（交叉淡化）": "crossfade",
    "Dissolve（溶解）": "dissolve",
    "Slide Left（向左滑動）": "slideleft",
    "Slide Right（向右滑動）": "slideright",
    "Zoom（縮放）": "zoom",
}


# ------------------------------------------------------------
# Step 1 的功能：處理使用者上傳的 PPT
# ------------------------------------------------------------
def handle_upload_notice(file_or_files):
    """
    純 UI 用的輔助函式：使用者上傳檔案後，顯示「✅ 已成功上傳：檔名」的提示文字。
    不涉及任何業務邏輯，只是回傳一段顯示文字，方便所有上傳元件共用。
    """
    if not file_or_files:
        return ""

    if isinstance(file_or_files, list):
        names = [os.path.basename(f.name if hasattr(f, "name") else f) for f in file_or_files]
        if not names:
            return ""
        if len(names) == 1:
            return f"✅ 已成功上傳：{names[0]}"
        return f"✅ 已成功上傳 {len(names)} 個檔案：{ '、'.join(names) }"

    name = os.path.basename(file_or_files.name if hasattr(file_or_files, "name") else file_or_files)
    return f"✅ 已成功上傳：{name}"


def handle_ppt_upload(ppt_file, custom_images_files):
    """
    當使用者上傳 PPT 並按下「解析 PPT」按鈕時執行。

    參數:
        ppt_file: 使用者上傳的 PPT 檔案，一定需要（用來解析文字、產生講稿）
        custom_images_files: 選填，使用者自行從 PowerPoint 匯出的投影片圖片（可多選）。
            如果有提供，「畫面」會直接使用這些圖片，不再透過 LibreOffice 轉換，
            因為 PowerPoint 自己匯出的圖片保證跟原始排版一模一樣，不會有跑版問題；
            PPT 檔案本身則仍然只用來解析文字/備忘稿，供後面產生講稿使用。
            圖片數量必須跟 PPT 頁數一致，才會採用；數量不符會改用自動轉換的畫面並提示原因。

    回傳 (依序對應到 Gradio 介面上的元件):
        1. summary_output: 文字說明，顯示共幾頁
        2. gallery_output: 每頁投影片圖片
        3. preview_output: 文字/備忘稿預覽 (Markdown)
        4. log_output: 執行紀錄
        5. slides_state: 存到 gr.State，供 Step2 講稿生成使用
        6. images_state: 每頁投影片圖片路徑清單，存到 gr.State，供 Step5 影片合成使用
        7. generate_script_button 的啟用狀態：解析成功才會啟用下一步的按鈕，
           引導使用者依序操作，避免不知道該按哪裡。
    """
    clear_log()

    if ppt_file is None:
        return "請先上傳 PPT 檔案", [], "", get_log_text(), None, None, gr.update(interactive=False)

    original_path = ppt_file.name if hasattr(ppt_file, "name") else ppt_file
    filename = safe_filename(os.path.basename(original_path))

    saved_ppt_path = os.path.join(DIR_PPT, filename)
    with open(original_path, "rb") as src, open(saved_ppt_path, "wb") as dst:
        dst.write(src.read())

    log(f"已收到上傳檔案：{filename}")

    try:
        # 文字內容一律從 PPT 本身解析（講稿要用），跟畫面圖片的來源分開處理。
        slides_text = read_ppt_slides(saved_ppt_path)
        slide_count = len(slides_text)

        images = None
        if custom_images_files:
            custom_paths = [f.name if hasattr(f, "name") else f for f in custom_images_files]
            custom_paths = sorted(custom_paths, key=natural_sort_key)

            if len(custom_paths) == slide_count:
                images = custom_paths
                log(
                    f"✅ 已採用您上傳的 {len(images)} 張投影片圖片作為影片畫面"
                    "（跳過自動轉換，不會有跑版問題）"
                )
            else:
                log(
                    f"⚠️ 您上傳的投影片圖片數量({len(custom_paths)}) 與 PPT 頁數"
                    f"({slide_count}) 不一致，已改用自動轉換的畫面。"
                    "請確認每一頁都有對應匯出一張圖片、且沒有多餘檔案。",
                    "warning",
                )

        if images is None:
            session_image_dir = make_session_dir(DIR_IMAGES)
            images = convert_ppt_to_images(saved_ppt_path, session_image_dir)

    except Exception as e:
        log(f"❌ 解析失敗：{e}", "error")
        return f"解析失敗：{e}", [], "", get_log_text(), None, None, gr.update(interactive=False)

    preview_lines = []
    for s in slides_text:
        preview_lines.append(f"### 第 {s['slide_index']} 頁：{s['title'] or '(無標題)'}")
        if s["body_text"]:
            preview_lines.append(f"**內文：** {s['body_text']}")
        if s["notes"]:
            preview_lines.append(f"**備忘稿：** {s['notes']}")
        preview_lines.append("---")
    preview_text = "\n\n".join(preview_lines)

    summary_text = f"✅ 解析完成，共 {slide_count} 頁投影片。可以繼續下方「產生講稿」步驟了。"

    # slides_text / images 分別存進 gr.State，供後面的講稿生成、影片合成使用（不用重新解析 PPT）
    return summary_text, images, preview_text, get_log_text(), slides_text, images, gr.update(interactive=True)


# ------------------------------------------------------------
# Step 2 的功能：依模式產生講稿
# ------------------------------------------------------------
def toggle_mode_inputs(mode_label):
    """
    根據使用者選擇的旁白模式，決定要顯示/隱藏哪些輸入欄位與提示文字：
        - 模式一 (own)      ：顯示 講稿上傳 + 講稿格式提示
        - 模式二 (ai_auto)  ：顯示 API Key + 參考資料上傳 + AI提示
        - 模式三 (notes)    ：都不顯示（直接用備忘稿）
        - 模式四 (ai_polish)：顯示 講稿上傳 + API Key + 參考資料上傳 + 兩種提示
    """
    mode = MODE_LABEL_TO_CODE.get(mode_label, "notes")
    show_script_upload = mode in ("own", "ai_polish")
    show_api_key = mode in ("ai_auto", "ai_polish")
    show_reference = mode in ("ai_auto", "ai_polish")
    return (
        gr.update(visible=show_script_upload),
        gr.update(visible=show_api_key),
        gr.update(visible=show_reference),
        gr.update(visible=show_script_upload),  # script_upload_hint
        gr.update(visible=show_api_key),  # ai_hint
    )


def handle_generate_script(mode_label, slides_state, script_file, api_key, reference_files):
    """
    當使用者按下「生成講稿」按鈕時執行。

    參數:
        mode_label: 使用者在 Radio 選單選擇的中文標籤
        slides_state: Step1 存下來的投影片文字資料 (list[dict])
        script_file: 使用者上傳的講稿檔案（模式一/四才需要）
        api_key: Gemini API Key（模式二/四才需要）
        reference_files: 使用者上傳的參考資料檔案清單（模式二/四可選填）。
                         有上傳的話，AI 生成/修飾講稿時會依據這些真實資料補充細節。

    回傳:
        1. script_preview: 可編輯的講稿全文（含頁碼標記）
        2. log_output: 執行紀錄
        3. generate_audio_button 的啟用狀態：成功產生講稿後才啟用「產生配音」按鈕
    """
    clear_log()

    if not slides_state:
        return "請先完成上方「Step 1：解析 PPT」，再產生講稿。", get_log_text(), gr.update(interactive=False)

    mode = MODE_LABEL_TO_CODE.get(mode_label)
    script_path = None
    if script_file is not None:
        script_path = script_file.name if hasattr(script_file, "name") else script_file

    reference_paths = []
    if reference_files:
        # gr.File(file_count="multiple") 回傳的是一個 list，每個元素有 .name 屬性
        for f in reference_files:
            reference_paths.append(f.name if hasattr(f, "name") else f)

    try:
        scripts = generate_all_scripts(
            mode=mode,
            slides_text=slides_state,
            uploaded_script_path=script_path,
            api_key=api_key if api_key else None,
            reference_paths=reference_paths,
        )
    except Exception as e:
        log(f"❌ 講稿生成失敗：{e}", "error")
        return f"❌ 講稿生成失敗：{e}", get_log_text(), gr.update(interactive=False)

    display_text = format_scripts_for_display(slides_state, scripts)
    return display_text, get_log_text(), gr.update(interactive=True)


def handle_generate_audio(
    voice_label, rate_percent, volume_percent, script_preview_text, slides_state,
    progress=gr.Progress(),
):
    """
    當使用者按下「🔊 產生配音」按鈕時執行。
    會讀取目前講稿編輯框中的內容（使用者可能已手動修改過），
    依序把每一頁講稿轉成語音檔。

    參數:
        voice_label: 使用者在下拉選單選擇的中文語音說明（對應 VOICE_OPTIONS 的 key）
        rate_percent / volume_percent: 語速/音量調整（Gradio Slider 的數值）
        script_preview_text: 講稿編輯框目前的文字內容
        slides_state: Step1 存下來的投影片資料，用來確認頁數
        progress: Gradio 內建的進度條物件（UI用），會呼叫 tts.py 原本就支援的
                  progress_callback 參數來即時回報「第幾頁/共幾頁」的生成進度，
                  純粹是畫面上的進度顯示，不影響任何配音生成的邏輯本身。

    回傳:
        1. audio_summary_output: 每頁時長的 Markdown 表格總覽
        2. log_output: 執行紀錄
        3. audio_state: 每頁音檔資訊清單 (存入 gr.State，供試聽/下一步使用)
        4. generate_subtitle_button 的啟用狀態
        5. generate_video_button 的啟用狀態
           （配音成功產生後，才能繼續「產生字幕」與「合成影片」）
    """
    clear_log()

    if not slides_state:
        return "請先完成 Step 1：解析 PPT。", get_log_text(), None, gr.update(interactive=False), gr.update(interactive=False)

    if not script_preview_text or not script_preview_text.strip():
        return "請先完成 Step 2：產生講稿。", get_log_text(), None, gr.update(interactive=False), gr.update(interactive=False)

    scripts = parse_scripts_from_display(script_preview_text, len(slides_state))
    voice = VOICE_OPTIONS.get(voice_label, "zh-TW-HsiaoChenNeural")
    session_audio_dir = make_session_dir(DIR_AUDIO)

    progress(0, desc="🎤 正在生成配音...")

    def _report_progress(fraction: float):
        percent = int(fraction * 100)
        progress(fraction, desc=f"🎤 正在生成配音... {percent}%")

    try:
        audio_results = generate_audio_for_all_slides(
            scripts,
            session_audio_dir,
            voice=voice,
            rate_percent=int(rate_percent),
            volume_percent=int(volume_percent),
            progress_callback=_report_progress,
        )
    except Exception as e:
        log(f"❌ 配音生成失敗：{e}", "error")
        return f"❌ 配音生成失敗：{e}", get_log_text(), None, gr.update(interactive=False), gr.update(interactive=False)

    progress(1.0, desc="✅ 配音生成完成")
    summary = format_audio_summary(audio_results)
    return summary, get_log_text(), audio_results, gr.update(interactive=True), gr.update(interactive=True)


def handle_preview_audio(slide_number, audio_state):
    """
    當使用者按下「▶️ 試聽」按鈕時執行，回傳指定頁碼的音檔路徑供播放。
    """
    clear_log()

    if not audio_state:
        return None, "請先按上方「🔊 產生配音」按鈕產生音檔。"

    try:
        idx = int(slide_number) - 1
    except (TypeError, ValueError):
        return None, "頁碼請輸入數字。"

    if idx < 0 or idx >= len(audio_state):
        return None, f"頁碼超出範圍（目前共 {len(audio_state)} 頁）。"

    return audio_state[idx]["audio_path"], get_log_text()


def handle_generate_subtitle(script_preview_text, slides_state, audio_state):
    """
    當使用者按下「🔤 產生字幕」按鈕時執行。
    依照目前講稿內容 + Step3 產生的每頁音檔時長，產生整份課程的 SRT 字幕，
    並存成 Output/字幕.srt 供下載。

    參數:
        script_preview_text: 講稿編輯框目前的文字內容
        slides_state: Step1 存下來的投影片資料，用來確認頁數
        audio_state: Step3 產生的每頁音檔資訊清單（含時長），用來對齊字幕時間軸

    回傳:
        1. subtitle_preview_output: SRT 內容預覽（過長時只顯示部分）
        2. subtitle_download: 可下載的 字幕.srt 檔案路徑
        3. log_output: 執行紀錄
    """
    clear_log()

    if not slides_state:
        return "", None, "請先完成 Step 1：解析 PPT。"

    if not script_preview_text or not script_preview_text.strip():
        return "", None, "請先完成 Step 2：產生講稿。"

    if not audio_state:
        return "", None, "請先完成 Step 3：產生配音（字幕時間需要依據配音實際長度）。"

    scripts = parse_scripts_from_display(script_preview_text, len(slides_state))

    try:
        srt_content = generate_full_srt(scripts, audio_state)
    except Exception as e:
        log(f"❌ 字幕生成失敗：{e}", "error")
        return "", None, get_log_text()

    output_path = os.path.join(DIR_OUTPUT, "字幕.srt")
    save_srt_file(srt_content, output_path)

    preview = srt_content
    if len(preview) > 3000:
        preview = preview[:3000] + "\n...(內容過長，僅顯示部分預覽，完整內容請下載檔案查看)"

    return preview, output_path, get_log_text()


def handle_generate_video(
    transition_label, transition_duration, include_subtitles, subtitle_font_size,
    logo_file, bgm_file, bgm_volume_percent, intro_file, outro_file,
    output_filename,
    script_preview_text, slides_state, images_state, audio_state,
    progress=gr.Progress(),
):
    """
    當使用者按下「🎬 合成影片」按鈕時執行。
    把 Step1 的投影片圖片 + Step3 的配音（依需要燒錄字幕）合成完整課程影片，
    並可選擇性加入 Step6 的 Logo / 背景音樂 / 片頭片尾。

    參數:
        transition_label: 使用者選擇的轉場效果中文標籤
        transition_duration: 使用者設定的轉場時間長度（秒）。
                              這個數值必須「同時」用在字幕時間軸計算 (generate_full_srt)
                              與影片合成 (build_full_course_video) 上，兩邊算出來的累積
                              時間點才會一致，否則字幕會跟畫面/配音對不齊。
        include_subtitles: 是否把字幕燒錄進影片畫面
        subtitle_font_size: 字幕字級大小（使用者可調整）
        logo_file: 選填，使用者上傳的 Logo 圖片
        bgm_file: 選填，使用者上傳的背景音樂
        bgm_volume_percent: 背景音樂音量（0~100）
        intro_file / outro_file: 選填，使用者上傳的片頭 / 片尾影片
        output_filename: 使用者自訂的輸出檔名（不含副檔名），預設為「課程」。
                         會自動清理不合法字元，並確保加上 .mp4 副檔名。
        script_preview_text: 講稿編輯框目前的文字內容（用來「即時」重新計算字幕，
                              確保字幕內容/時間軸一定跟這次要合成的配音完全對應，
                              不會發生「拿到之前產生、但後來又改過講稿或重新配音」的舊字幕）
        slides_state: Step1 存下來的投影片資料，用來確認頁數
        images_state: Step1 存下來的每頁投影片圖片路徑清單
        audio_state: Step3 產生的每頁音檔資訊清單
        progress: Gradio 內建的進度條物件（UI用），串接 video.py 既有的 progress_callback 參數

    回傳:
        1. video_preview_output: 合成好的影片路徑（供 gr.Video 預覽播放）
        2. video_download: 合成好的影片路徑（供下載）
        3. log_output: 執行紀錄
    """
    clear_log()

    if not slides_state or not images_state:
        return None, None, "請先完成 Step 1：解析 PPT。"

    if not audio_state:
        return None, None, "請先完成 Step 3：產生配音（影片每頁停留時間依配音長度決定）。"

    if len(images_state) != len(audio_state):
        log(
            f"⚠️ 投影片頁數({len(images_state)}) 與配音數量({len(audio_state)}) 不一致，"
            "可能是先解析了新的 PPT、但還沒重新產生配音，請確認後重新整套跑一次"
            "（Step1→Step2→Step3→...），避免內容對不上頁數。",
            "warning",
        )

    transition = TRANSITION_LABEL_TO_CODE.get(transition_label, "fade")
    try:
        transition_duration = float(str(transition_duration).replace("秒", "").strip())
    except (TypeError, ValueError):
        transition_duration = DEFAULT_TRANSITION_DURATION

    # 重要：字幕改成「這裡即時重新計算」，而不是讀取 Step4 另外存檔的 字幕.srt。
    # 這樣可以保證字幕的文字內容與時間軸，一定是根據「這次真正要拿去合成影片」的
    # script_preview_text 與 audio_state 算出來的，不會因為使用者中途改過講稿、
    # 重新配音、卻忘記回頭重按一次「產生字幕」，而燒錄進一份跟畫面對不上的舊字幕。
    # transition_duration 也要傳進去，確保字幕的累積時間軸把轉場緩衝時間算進去，
    # 跟影片實際畫面切換的時間點一致。
    srt_path = None
    if include_subtitles:
        if not script_preview_text or not script_preview_text.strip():
            log("⚠️ 講稿內容是空的，這次先輸出不含字幕的影片。", "warning")
        else:
            scripts_for_srt = parse_scripts_from_display(script_preview_text, len(slides_state))
            try:
                srt_content = generate_full_srt(
                    scripts_for_srt, audio_state, transition_duration=transition_duration,
                )
                srt_path = os.path.join(DIR_OUTPUT, "字幕.srt")
                save_srt_file(srt_content, srt_path)
            except Exception as e:
                log(f"⚠️ 字幕即時重算失敗，這次先輸出不含字幕的影片：{e}", "warning")
                srt_path = None

    def _resolve_upload_path(f):
        if f is None:
            return None
        return f.name if hasattr(f, "name") else f

    logo_path = _resolve_upload_path(logo_file)
    bgm_path = _resolve_upload_path(bgm_file)
    intro_path = _resolve_upload_path(intro_file)
    outro_path = _resolve_upload_path(outro_file)

    filename_base = safe_filename((output_filename or "").strip()) or "課程"
    if filename_base.lower().endswith(".mp4"):
        filename_base = filename_base[:-4]
    output_path = os.path.join(DIR_OUTPUT, f"{filename_base}.mp4")

    progress(0, desc="🎬 正在合成影片...")

    def _report_progress(fraction: float):
        percent = int(fraction * 100)
        progress(fraction, desc=f"🎬 正在合成影片... {percent}%")

    try:
        build_full_course_video(
            images_state,
            audio_state,
            output_path,
            transition_duration=transition_duration,
            srt_path=srt_path,
            subtitle_font_size=int(subtitle_font_size),
            transition=transition,
            logo_path=logo_path,
            bgm_path=bgm_path,
            bgm_volume=(bgm_volume_percent or 0) / 100,
            intro_path=intro_path,
            outro_path=outro_path,
            progress_callback=_report_progress,
        )
    except Exception as e:
        log(f"❌ 影片合成失敗：{e}", "error")
        return None, None, get_log_text()

    progress(1.0, desc="✅ 影片合成完成")
    return output_path, output_path, get_log_text()


def handle_save_script(script_preview_text, slides_state):
    """
    當使用者按下「儲存講稿為 Word」按鈕時執行。
    會把目前編輯框中的文字（使用者可能已手動修改過）重新解析回每頁講稿，
    再輸出成 Output/講稿.docx，讓使用者下載。
    """
    clear_log()

    if not slides_state:
        return None, "請先完成 Step 1 與 Step 2，再儲存講稿。"

    if not script_preview_text or not script_preview_text.strip():
        return None, "講稿內容是空的，請先產生或輸入講稿。"

    scripts = parse_scripts_from_display(script_preview_text, len(slides_state))
    output_path = os.path.join(DIR_OUTPUT, "講稿.docx")

    try:
        save_scripts_to_docx(slides_state, scripts, output_path)
    except Exception as e:
        log(f"❌ 儲存講稿失敗：{e}", "error")
        return None, get_log_text()

    return output_path, get_log_text()


# ------------------------------------------------------------
# 建立 Gradio 介面
# ------------------------------------------------------------
def build_ui():
    """
    建立並回傳 Gradio Blocks 介面物件。

    視覺風格（主題色、卡片、動畫、使用手冊）統一定義在 ui_theme.py，
    這裡只負責「組裝」畫面結構與串接既有的後端函式，不包含任何樣式邏輯，
    也完全沒有更動任何 PPT 解析 / 講稿生成 / 配音 / 字幕 / 影片合成的後端流程。
    """
    with gr.Blocks(title="PPT2Course AI", theme=build_theme(), css=CUSTOM_CSS) as demo:
        gr.Markdown(
            """
            # 🎓 PPT2Course Lab
            ### 一鍵將 PowerPoint 轉換成完整線上課程影片
            """
        )
        gr.HTML(STEPS_BAR_HTML)

        if not ENV_STATUS["all_ok"]:
            missing = []
            if not ENV_STATUS["libreoffice"]:
                missing.append("LibreOffice")
            if not ENV_STATUS["ffmpeg"]:
                missing.append("FFmpeg")
            gr.Markdown(
                f"⚠️ **注意：系統缺少以下工具，部分功能可能無法使用：{', '.join(missing)}**"
            )

        # gr.State：在使用者操作過程中，暫存 Step1 解析出來的投影片資料，
        # 讓 Step2 可以直接使用，不需要重新解析 PPT。
        slides_state = gr.State(value=None)
        # gr.State：暫存 Step3 產生的每頁音檔資訊（路徑/時長），供試聽與後續步驟使用。
        audio_state = gr.State(value=None)
        # gr.State：暫存 Step1 解析出來的每頁投影片圖片路徑，供 Step5 影片合成使用。
        images_state = gr.State(value=None)

        # ---------------- Step 1：上傳 PPT ----------------
        with gr.Group(elem_classes=["ptc-card"]):
            gr.Markdown('<div class="ptc-step-title">1️⃣ 上傳並解析 PPT</div>')
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('<div class="ptc-upload-title">📑 PPT 檔案（必要）</div>')
                    ppt_input = gr.File(
                        label="📄 點擊或拖曳檔案至此｜.pptx",
                        file_types=[".pptx", ".ppt"],
                        elem_classes=["ptc-upload"],
                    )
                    ppt_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])

                    gr.Markdown('<div class="ptc-upload-title">🖼 投影片圖片（選填，避免跑版）</div>')
                    custom_images_input = gr.File(
                        label="📄 點擊或拖曳檔案至此｜可多選 png/jpg",
                        file_types=[".png", ".jpg", ".jpeg"],
                        file_count="multiple",
                        elem_classes=["ptc-upload"],
                    )
                    custom_images_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])

                    parse_button = gr.Button("🔍 解析 PPT", variant="primary")
                    summary_output = gr.Textbox(label="解析結果摘要", interactive=False)

                with gr.Column(scale=2):
                    gallery_output = gr.Gallery(
                        label="投影片預覽", columns=4, height="auto"
                    )

            gr.Markdown(
                "💡 **想避免 PPT 轉換跑版？** 建議在 PowerPoint 中「檔案 → 匯出 → 變更檔案類型 → "
                "PNG」，把每一頁另存成圖片，跟 PPT 檔案一起上傳。畫面會直接採用您匯出的圖片"
                "（PowerPoint 自己輸出的圖片，保證排版跟原始投影片一模一樣），"
                "PPT 檔案則仍會用來解析文字、產生講稿。圖片數量需與投影片頁數一致才會採用。",
                elem_classes=["ptc-hint"],
            )

            with gr.Accordion("📄 投影片文字內容預覽", open=False):
                preview_output = gr.Markdown()

        # ---------------- Step 2：產生講稿 ----------------
        with gr.Group(elem_classes=["ptc-card"]):
            gr.Markdown('<div class="ptc-step-title">2️⃣ 產生講稿</div>'
                        '<div class="ptc-step-sub">請選擇一種旁白模式</div>')

            mode_radio = gr.Radio(
                choices=list(MODE_LABEL_TO_CODE.keys()),
                value="③ 使用 PowerPoint 備忘稿 (Speaker Notes)",
                label="旁白模式",
            )

            with gr.Row():
                with gr.Column():
                    gr.Markdown('<div class="ptc-upload-title">📝 Word / TXT 講稿（模式①④需要）</div>')
                    script_upload = gr.File(
                        label="📄 點擊或拖曳檔案至此｜.docx / .txt",
                        file_types=[".docx", ".txt"],
                        visible=False,
                        elem_classes=["ptc-upload"],
                    )
                    script_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])
                api_key_input = gr.Textbox(
                    label="🔑 Gemini API Key｜模式②④ 需要",
                    type="password",
                    placeholder="在此貼上您的 Gemini API Key",
                    visible=False,
                )

            gr.Markdown('<div class="ptc-upload-title">📚 參考資料（選填）</div>', elem_classes=["ptc-hint"])
            reference_upload = gr.File(
                label="📄 點擊或拖曳檔案至此｜可多選 PDF / docx / txt",
                file_types=[".pdf", ".docx", ".txt"],
                file_count="multiple",
                visible=False,
                elem_classes=["ptc-upload"],
            )
            reference_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])

            script_upload_hint = gr.Markdown(
                "💡 建議在每頁講稿前加上「第1頁」「第2頁」等標記，確保正確對應投影片。",
                elem_classes=["ptc-hint"],
                visible=False,
            )
            ai_hint = gr.Markdown(
                "💡 還沒有 API Key？[點此免費申請](https://aistudio.google.com/app/apikey)"
                "　｜　想讓內容更真實？可在下方上傳參考資料（法規全文、政策文件等）",
                elem_classes=["ptc-hint"],
                visible=False,
            )

            generate_script_button = gr.Button("✍️ 生成講稿", variant="primary", interactive=False)

            script_preview = gr.Textbox(
                label="講稿預覽 / 編輯（可直接修改文字，修改後請按下方「儲存」）",
                lines=18,
            )

            save_script_button = gr.Button("💾 儲存講稿為 Word 檔")
            script_download = gr.DownloadButton("⬇️ 下載 講稿.docx", variant="secondary", elem_classes=["ptc-download"])

        # ---------------- Step 3：AI 配音 ----------------
        with gr.Group(elem_classes=["ptc-card"]):
            gr.Markdown('<div class="ptc-step-title">3️⃣ 生成 配音</div>'
                        '<div class="ptc-step-sub">使用免費的 Edge-TTS</div>')

            with gr.Row():
                voice_dropdown = gr.Dropdown(
                    choices=list(VOICE_OPTIONS.keys()),
                    value=DEFAULT_VOICE_LABEL,
                    label="🎙️ 選擇配音聲音",
                )
                rate_slider = gr.Slider(
                    minimum=-50, maximum=50, value=0, step=5,
                    label="語速調整 (%)｜負值變慢，正值變快",
                )
                volume_slider = gr.Slider(
                    minimum=-50, maximum=50, value=0, step=5,
                    label="音量調整 (%)｜負值變小聲，正值變大聲",
                )

            generate_audio_button = gr.Button("🔊 產生配音", variant="primary", interactive=False)
            audio_summary_output = gr.Markdown(label="配音結果總覽")

            with gr.Row():
                preview_slide_number = gr.Number(
                    label="想試聽第幾頁的配音？", value=1, precision=0, minimum=1,
                )
                preview_button = gr.Button("▶️ 試聽這一頁")
            preview_audio_output = gr.Audio(label="配音預覽播放")

        # ---------------- Step 4：產生字幕 ----------------
        with gr.Group(elem_classes=["ptc-card"]):
            gr.Markdown('<div class="ptc-step-title">4️⃣ 產生字幕</div>'
                        '<div class="ptc-step-sub">SRT 格式，會依配音時長自動對齊時間軸</div>')

            generate_subtitle_button = gr.Button("🔤 產生字幕", variant="primary", interactive=False)
            subtitle_preview_output = gr.Textbox(
                label="字幕預覽（SRT 格式）", lines=14, interactive=False,
            )
            subtitle_download = gr.DownloadButton("⬇️ 下載 字幕.srt", variant="secondary", elem_classes=["ptc-download"])

        # ---------------- Step 5：影片合成（含 Step 6：Logo / 背景音樂 / 片頭片尾） ----------------
        with gr.Group(elem_classes=["ptc-card"]):
            gr.Markdown('<div class="ptc-step-title">5️⃣ 合成課程影片</div>'
                        '<div class="ptc-step-sub">輸出成 MP4，可選擇轉場效果、Logo、背景音樂、片頭片尾</div>')

            with gr.Row():
                transition_dropdown = gr.Dropdown(
                    choices=list(TRANSITION_LABEL_TO_CODE.keys()),
                    value="Fade（淡出淡入）",
                    label="🎞️ 轉場效果",
                )
                transition_duration_dropdown = gr.Dropdown(
                    choices=[f"{s} 秒" for s in TRANSITION_DURATION_CHOICES],
                    value=f"{DEFAULT_TRANSITION_DURATION} 秒",
                    label="⏱️ 轉場時間",
                )
                include_subtitles_checkbox = gr.Checkbox(
                    label="🔤 把字幕燒錄進影片畫面",
                    value=True,
                )
                subtitle_font_size_slider = gr.Slider(
                    minimum=14, maximum=44, value=22, step=2,
                    label="字幕大小",
                )

            with gr.Accordion("🖼️ 進階選項：Logo・背景音樂・片頭片尾（全部選填）", open=False):
                gr.Markdown('<div class="ptc-upload-title">🖼 Logo 圖片</div>')
                with gr.Row():
                    logo_upload = gr.File(
                        label="📄 點擊或拖曳檔案至此｜png/jpg，固定顯示在影片右上角",
                        file_types=[".png", ".jpg", ".jpeg"],
                        elem_classes=["ptc-upload"],
                    )
                logo_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])

                gr.Markdown('<div class="ptc-upload-title">🎵 背景音樂</div>')
                with gr.Row():
                    bgm_upload = gr.File(
                        label="📄 點擊或拖曳檔案至此｜mp3，自動循環並與旁白混音",
                        file_types=[".mp3", ".wav", ".m4a"],
                        elem_classes=["ptc-upload"],
                    )
                    bgm_volume_slider = gr.Slider(
                        minimum=0, maximum=100, value=20, step=5,
                        label="背景音樂音量 (%)",
                    )
                bgm_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])

                gr.Markdown('<div class="ptc-upload-title">🎬 片頭 / 片尾影片</div>')
                with gr.Row():
                    intro_upload = gr.File(
                        label="📄 點擊或拖曳檔案至此｜片頭 mp4，接在課程最前面",
                        file_types=[".mp4", ".mov", ".mkv"],
                        elem_classes=["ptc-upload"],
                    )
                    outro_upload = gr.File(
                        label="📄 點擊或拖曳檔案至此｜片尾 mp4，接在課程最後面",
                        file_types=[".mp4", ".mov", ".mkv"],
                        elem_classes=["ptc-upload"],
                    )
                with gr.Row():
                    intro_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])
                    outro_upload_notice = gr.Markdown("", elem_classes=["ptc-upload-notice"])

            output_filename_input = gr.Textbox(
                label="📝 輸出檔名（不需輸入副檔名，會自動加上 .mp4）",
                value="課程",
                placeholder="例如：職場霸凌教育訓練",
            )

            generate_video_button = gr.Button("🎬 合成影片", variant="primary", interactive=False)
            gr.Markdown(
                "💡 頁數較多、或有加入片頭片尾/背景音樂時，合成過程可能需要幾分鐘，請耐心等待。",
                elem_classes=["ptc-hint"],
            )

            video_preview_output = gr.Video(label="課程影片預覽")
            video_download = gr.DownloadButton("⬇️ 下載完整課程影片", variant="primary", elem_classes=["ptc-download"])

        # ---------------- 共用：執行紀錄 ----------------
        with gr.Accordion("📋 即時執行紀錄 (Log)", open=False):
            log_output = gr.Textbox(label="Log", lines=10, interactive=False)

        # ---------------- 懸浮使用手冊按鈕 ----------------
        gr.HTML(HELP_FAB_HTML)

        # ---------------- 事件綁定 ----------------
        # 上傳成功提示（純UI，不影響任何後端邏輯）
        ppt_input.upload(fn=handle_upload_notice, inputs=[ppt_input], outputs=[ppt_upload_notice])
        custom_images_input.upload(
            fn=handle_upload_notice, inputs=[custom_images_input], outputs=[custom_images_upload_notice],
        )
        script_upload.upload(fn=handle_upload_notice, inputs=[script_upload], outputs=[script_upload_notice])
        reference_upload.upload(
            fn=handle_upload_notice, inputs=[reference_upload], outputs=[reference_upload_notice],
        )
        logo_upload.upload(fn=handle_upload_notice, inputs=[logo_upload], outputs=[logo_upload_notice])
        bgm_upload.upload(fn=handle_upload_notice, inputs=[bgm_upload], outputs=[bgm_upload_notice])
        intro_upload.upload(fn=handle_upload_notice, inputs=[intro_upload], outputs=[intro_upload_notice])
        outro_upload.upload(fn=handle_upload_notice, inputs=[outro_upload], outputs=[outro_upload_notice])

        parse_button.click(
            fn=handle_ppt_upload,
            inputs=[ppt_input, custom_images_input],
            outputs=[
                summary_output, gallery_output, preview_output, log_output,
                slides_state, images_state, generate_script_button,
            ],
        )

        mode_radio.change(
            fn=toggle_mode_inputs,
            inputs=[mode_radio],
            outputs=[script_upload, api_key_input, reference_upload, script_upload_hint, ai_hint],
        )

        generate_script_button.click(
            fn=handle_generate_script,
            inputs=[mode_radio, slides_state, script_upload, api_key_input, reference_upload],
            outputs=[script_preview, log_output, generate_audio_button],
        )

        save_script_button.click(
            fn=handle_save_script,
            inputs=[script_preview, slides_state],
            outputs=[script_download, log_output],
        )

        generate_audio_button.click(
            fn=handle_generate_audio,
            inputs=[voice_dropdown, rate_slider, volume_slider, script_preview, slides_state],
            outputs=[
                audio_summary_output, log_output, audio_state,
                generate_subtitle_button, generate_video_button,
            ],
        )

        preview_button.click(
            fn=handle_preview_audio,
            inputs=[preview_slide_number, audio_state],
            outputs=[preview_audio_output, log_output],
        )

        generate_subtitle_button.click(
            fn=handle_generate_subtitle,
            inputs=[script_preview, slides_state, audio_state],
            outputs=[subtitle_preview_output, subtitle_download, log_output],
        )

        generate_video_button.click(
            fn=handle_generate_video,
            inputs=[
                transition_dropdown, transition_duration_dropdown,
                include_subtitles_checkbox, subtitle_font_size_slider,
                logo_upload, bgm_upload, bgm_volume_slider, intro_upload, outro_upload,
                output_filename_input,
                script_preview, slides_state, images_state, audio_state,
            ],
            outputs=[video_preview_output, video_download, log_output],
        )

    return demo


# ------------------------------------------------------------
# 主程式進入點
# ------------------------------------------------------------
if __name__ == "__main__":
    demo = build_ui()
    demo.launch(share=True)  # share=True 會產生一組公開網址，方便在 Colab 使用


Writing app.py


In [12]:
import os
for d in ['PPT', 'Script', 'Audio', 'Images', 'Output', 'Assets']:
    os.makedirs(d, exist_ok=True)
print('✅ 資料夾建立完成')


✅ 資料夾建立完成


In [13]:
%run app.py


/content/app.py:558: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="PPT2Course AI", theme=build_theme(), css=CUSTOM_CSS) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb1736c38704ada787.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
